In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from estimate_level_adjustment.adjustment_model import (
    em_latent_normal,
)

from estimate_level_adjustment.utils import (
    coverage_calibration_curve,
    leave_one_out_validation,
)

# Vanilla PPI estimator
from estimate_level_adjustment.cov_shift_estimator import CovShiftPPIEstimator

#Adjusted PPI estimator
from estimate_level_adjustment.adjusted_ppi_estimator import AdjustedCovShiftPPIEstimator

# New data generator
from estimate_level_adjustment.data_generators import (
    CovShiftDataGenerator,
)


from estimate_level_adjustment.stats_utils import conf_to_z, get_ci

# Generate data

# Simulation Setup: Covariate Shift and Concept Drift
## 1. Data Generating Process
**Number of covariates**:  $ P = 4.$

**Domains**: $S_i \in \{1, 2, \dots, K\}$.

**Covariates**:  For domain \(s\), $X_i \mid S_i = s \sim \mathcal{N}(\mu_s, I_4),$ where \($I_4$\) is the \($4 \times 4$\) identity matrix and \($\mu_s \in \mathbb{R}^4$\) is the domain-specific covariate mean.

**Outcome**: $Y_i \in \{0, 1\}.$

**Target estimand**:  The target is the mean prevalence in the target domain \($s = K$\): $\mathbb{E}[Y \mid S = K].$

**Domain Means**: Sampled from the uniform ball $||\mu_s||_2 \leq 2$ TODO: review how this is generated. 

**Target Mean**: The target domain mean is $\mu_s = (\frac{1}{2}, -\frac{1}{2}, \frac{1}{2}, -\frac{1}{2})$

---
## 2. Outcome Model
For a covariate vector \($x \in \mathbb{R}^P$\) and domain \($s$\), 
$\mathbb{P}(Y = 1 \mid X = x, S = s) = \left(1 + \exp\left\{- \lambda_1 \left(\sum_{p = 1}^P \mathbb{I}(x_p \geq \delta_s) - \frac{P}{2}\right) + \phi_1 \right\}\right)^{-1}$. 

### 2.1 Covariate Shift (No Concept Drift)
If $\delta_s = 0 \quad \text{for all } s$ then the conditional response model is the same across domains, and only the covariate distribution changes via \($\mu_s$\). This corresponds to a pure **covariate shift** scenario.

### 2.2 Concept Drift
To introduce **concept drift** across domains, let $\delta_s \sim \mathcal{N}(\kappa, \kappa^2/2)$ for domains \($s$\), where \($\kappa$\) controls the magnitude of deviation from pure covariate shift. When \($\kappa = 0$\), we recover the covariate shift setting. For replicability in the target parameter, the term $\delta_s$ for $s = K$ we set $\delta_s = \kappa$.  

---
## 3. Proxy Outcome Model
We also have access to a proxy for the true outcome, defined as a prediction function of \($x$\): $Y^* = f(x)$, where $ f(x) = \frac{1}{\pi}\arctan\left(\lambda_2 \left(\sum_{p = 1}^P \mathbb{I}(x_p \geq 0) - \frac{P}{2}\right) + \phi_2\right) + \frac{1}{2}.$

Properties:
- $f(x)$ is an approximation of the true conditional response $\mathbb{P}(Y = 1 \mid X = x, S)$.
- It **does not** account for domain-specific shifts \($\delta_s$\), i.e., it ignores concept drift and uses the threshold \($0$\) instead of \($\delta_s$\).
---
## 4. Methods Compared
We compare coverage and interval length for the following confidence interval procedures for the target prevalence \(\mathbb{E}[Y \mid S = K]\):
1. Proxy-only based confidence intervals  
2. Proxy-based confidence intervals with estimate-level adjustments  
3. Prediction Powered confidence intervals  
4. Prediction Powered confidence intervals with estimate-level adjustments  
5. Covariate-Shift Prediction Powered confidence intervals  
6. Covariate-Shift Prediction Powered confidence intervals with estimate-level adjustments  
(Here “estimate-level adjustments” refers to additional corrections applied at the level of the point estimate; details depend on the specific implementation.)
---
## 5. Covariate Shift and Importance Weights
To emphasize the additional error under covariate shift, we use **known importance weights**:
$w_s(x) = \frac{p_K(x)}{p_s(x)} = \exp\left(\frac{1}{2}\|x - \mu_s\|^2 - \frac{1}{2}\|x - \mu_K\|^2\right),$
where:
- $p_s(x)$ is the density of domain $s$,
- $p_K(x)$ is the density in the target domain $K$.

These weights reweight observations from domain \($s$\) so that their covariate distribution matches that of the target domain.

---
## 6. Simulation Design
- **Sample size per domain**
  $n_s \in \{100, 500, 1000, 5000, 10000\}.$
- **Number of domains**
  $K \in \{5, 10, 20, 30\}.$
- **Degree of concept drift**
  $\kappa \in \{0, 0.5,1, 5\}.$
- **Response Parameters**: $\lambda_1 = \frac{1}{2}, \phi_1 = 2$
- **Parameters Parameters**: $\lambda_2 = \frac{1}{2}, \phi_2 = 2$

For each combination of: $n_s, K, \kappa$,

We:
1. Generate data in each domain with $ X_i \mid S_i = s \sim \mathcal{N}(\mu_s, I_4)$ and $Y_i \sim \text{Bernoulli}\big(\mathbb{P}(Y = 1 \mid X_i, S_i = s)\big)$
   where the probability is given by the logistic model with domain-specific $\delta_s$.
2. Construct confidence intervals for $\mathbb{E}[Y \mid S = K]$ using the six methods above.
3. Repeat this process **1000** times.
4. Compute:
   - empirical coverage (proportion of CIs containing the true target prevalence),
   - average confidence interval length.

---
## 7. Approximating the True Target Prevalence
For the  target configuration, we approximate the true prevalence by sampling $10^7$ times. This Monte Carlo approximation is used as the “ground truth” for evaluating empirical coverage of the confidence intervals.




# Covariate Shift Coverage Simulation Study

This notebook evaluates the coverage of 95% confidence intervals for different estimation methods under varying degrees of concept drift (kappa) and sample sizes.

## Methods Compared
1. **Primary Estimate Only**: Uses only the labeled (primary) data
2. **Proxy Estimate Only**: Uses only the proxy predictions
3. **PPI (Prediction-Powered Inference)**: Combines primary and proxy data
4. **PPI with Weighting**: PPI with importance weighting for covariate shift correction

## Simulation Parameters
- Kappa values: 0, 0.001, 0.005, 0.01, 0.05, 0.1
- Sample sizes: 100 to 10,000 (logarithmically spaced)
- 3,000 simulation replications per combination
- True prevalence computed with 10,000,000 Monte Carlo samples

In [2]:
# (c) Meta Platforms, Inc. and affiliates. Confidential and proprietary.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Tuple, List, Dict, Any
import time
from pathlib import Path

from estimate_level_adjustment.data_generators import CovShiftDataGenerator
from estimate_level_adjustment.cov_shift_estimator import CovShiftPPIEstimator
from estimate_level_adjustment.stats_utils import Estimate

In [3]:
# =============================================================================
# SIMULATION CONFIGURATION
# =============================================================================


# Number of covariates
P = 4

# Target domain means (alternating 1/-1s, normalized)
mu_K = np.array([1.0 if i % 2 == 0 else -1.0 for i in range(P)])
mu_K = mu_K / np.linalg.norm(mu_K)

# Number of domains to evaluate (source + target)
N_DOMAINS_VALUES = [5, 10, 25]
# N_DOMAINS_VALUES = [5]  # TODO: remove

print(f"Number of domains (K): {N_DOMAINS_VALUES}")

# Primary and proxy model parameters
PRIMARY_PHI = P / 2  # = 2.0
PROXY_PHI = 1 + P / 2  # = 3.0
PRIMARY_LAMBDA = 0.5
PROXY_LAMBDA = 0.5

# Concept drift degrees (kappa values)
KAPPA_VALUES = [0, 1, 5]

# Sample sizes (logarithmically spaced from 100 to 10000)
# SAMPLE_SIZES = np.unique(np.logspace(np.log10(100), np.log10(10000), num=10).astype(int))
# SAMPLE_SIZES = np.array([100, 500, 1000, 5000, 10000, 50000]) #
SAMPLE_SIZES = np.array([500, 1000])  # Testing smaller Simulations Set

print(f"Sample sizes: {SAMPLE_SIZES}")

# Number of simulation replications
N_SIMULATIONS = 3000
# N_SIMULATIONS = 300  # Testing on a smaller scale


# Monte Carlo samples for true prevalence
N_MC_SAMPLES = 10_000_000

# Confidence level
CONFIDENCE_LEVEL = 0.95

# Random seed base for reproducibility
BASE_SEED = 44


# Number of bootstrap samples (reduced for speed)
N_BOOTSTRAP = 500


# Output directory for CSV files
OUTPUT_DIR = Path("/tmp/covariate_shift_simulation_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results will be saved to: {OUTPUT_DIR}")

Number of domains (K): [5, 10, 25]
Sample sizes: [ 500 1000]
Results will be saved to: /tmp/covariate_shift_simulation_results


In [4]:
@dataclass
class SimulationResult:
    """Result from a single simulation run."""
    kappa: float
    sample_size: int
    n_domains: int  # Number of domains (K)
    sim_id: int
    true_prevalence: float
    # Primary only estimates
    primary_estimate: float
    primary_lower: float
    primary_upper: float
    primary_covers: bool
    primary_ci_length: float
    # Proxy only estimates
    proxy_estimate: float
    proxy_lower: float
    proxy_upper: float
    proxy_covers: bool
    proxy_ci_length: float
    # PPI estimates (unweighted)
    ppi_estimate: float
    ppi_lower: float
    ppi_upper: float
    ppi_covers: bool
    ppi_ci_length: float
    # PPI weighted estimates
    ppi_weighted_estimate: float
    ppi_weighted_lower: float
    ppi_weighted_upper: float
    ppi_weighted_covers: bool
    ppi_weighted_ci_length: float
    # Proxy + Adjustment estimates (MoM)
    proxy_adj_estimate: float
    proxy_adj_lower: float
    proxy_adj_upper: float
    proxy_adj_covers: bool
    proxy_adj_gamma_squared: float
    proxy_adj_ci_length: float
    # PPI + Adjustment estimates (MoM)
    ppi_adj_estimate: float
    ppi_adj_lower: float
    ppi_adj_upper: float
    ppi_adj_covers: bool
    ppi_adj_gamma_squared: float
    ppi_adj_ci_length: float
    # PPI Weighted + Adjustment estimates (MoM)
    ppi_weighted_adj_estimate: float
    ppi_weighted_adj_lower: float
    ppi_weighted_adj_upper: float
    ppi_weighted_adj_covers: bool
    ppi_weighted_adj_gamma_squared: float
    ppi_weighted_adj_ci_length: float
    # Proxy + Domain Bootstrap estimates (estimate-level resampling)
    proxy_domain_boot_estimate: float
    proxy_domain_boot_lower: float
    proxy_domain_boot_upper: float
    proxy_domain_boot_covers: bool
    proxy_domain_boot_ci_length: float
    proxy_domain_boot_rho_mean: float
    proxy_domain_boot_gamma_sq_mean: float
    # PPI + Domain Bootstrap estimates (estimate-level resampling)
    ppi_domain_boot_estimate: float
    ppi_domain_boot_lower: float
    ppi_domain_boot_upper: float
    ppi_domain_boot_covers: bool
    ppi_domain_boot_ci_length: float
    ppi_domain_boot_rho_mean: float
    ppi_domain_boot_gamma_sq_mean: float
    # PPI Weighted + Domain Bootstrap estimates (estimate-level resampling)
    ppi_weighted_domain_boot_estimate: float
    ppi_weighted_domain_boot_lower: float
    ppi_weighted_domain_boot_upper: float
    ppi_weighted_domain_boot_covers: bool
    ppi_weighted_domain_boot_ci_length: float
    ppi_weighted_domain_boot_rho_mean: float
    ppi_weighted_domain_boot_gamma_sq_mean: float
    # Timing fields (in seconds)
    time_data_generation: float
    time_primary_estimate: float
    time_proxy_estimate: float
    time_ppi_estimate: float
    time_ppi_weighted_estimate: float
    time_proxy_adj_estimate: float
    time_ppi_adj_estimate: float
    time_ppi_weighted_adj_estimate: float
    time_proxy_domain_boot_estimate: float
    time_ppi_domain_boot_estimate: float
    time_ppi_weighted_domain_boot_estimate: float
    time_total: float

In [5]:
def compute_true_prevalence(kappa: float, n_domains: int, n_mc_samples: int = N_MC_SAMPLES) -> float:
    """
    Compute the true target population prevalence via Monte Carlo.

    Uses a large number of samples to accurately estimate the expected
    value of the primary outcome in the target domain.
    """
    generator = CovShiftDataGenerator(
        n_samples_per_domain=1,  # Not used for prevalence calculation
        n_domains=n_domains,
        concept_drift_degree=kappa,
        primary_lambda=PRIMARY_LAMBDA,
        primary_phi=PRIMARY_PHI,
        proxy_lambda=PROXY_LAMBDA,
        proxy_phi=PROXY_PHI,
        n_covariates=P,
        target_domain_means=mu_K,
        random_seed=BASE_SEED,
    )
    return generator.calculate_target_population_prevalence(n_samples=n_mc_samples)


# Pre-compute true prevalence for each (kappa, n_domains) combination
# Note: True prevalence only depends on kappa, not n_domains, since the target
# domain distribution is the same regardless of how many source domains we have
print("Computing true prevalence for each kappa value...")
TRUE_PREVALENCES = {}
for kappa in KAPPA_VALUES:
    start_time = time.time()
    # Use K=5 for prevalence calculation (result is same for all K)
    TRUE_PREVALENCES[kappa] = compute_true_prevalence(kappa, n_domains=5)
    elapsed = time.time() - start_time
    print(f"  kappa={kappa}: prevalence={TRUE_PREVALENCES[kappa]:.6f} (computed in {elapsed:.1f}s)")

print("\nTrue prevalences:")
for kappa, prev in TRUE_PREVALENCES.items():
    print(f"  kappa={kappa}: {prev:.6f}")

Computing true prevalence for each kappa value...
  kappa=0: prevalence=0.127594 (computed in 1.1s)
  kappa=1: prevalence=0.242878 (computed in 0.9s)
  kappa=5: prevalence=0.268751 (computed in 0.8s)

True prevalences:
  kappa=0: 0.127594
  kappa=1: 0.242878
  kappa=5: 0.268751


In [6]:
def run_single_simulation(
    kappa: float,
    sample_size: int,
    n_domains: int,
    sim_id: int,
    true_prevalence: float,
) -> SimulationResult:
    """
    Run a single simulation and compute estimates using CovShiftPPIEstimator
    and AdjustedCovShiftPPIEstimator.

    Methods computed:
    - Primary only (oracle baseline)
    - Proxy only
    - PPI (unweighted)
    - PPI (weighted)
    - Proxy + Adjustment (MoM)
    - PPI + Adjustment (MoM)
    - PPI Weighted + Adjustment (MoM)
    - Proxy + Domain Bootstrap (estimate-level resampling)
    - PPI + Domain Bootstrap (estimate-level resampling)
    - PPI Weighted + Domain Bootstrap (estimate-level resampling)

    Each method is timed separately to identify bottlenecks.
    """
    import time as timing_module

    total_start = timing_module.perf_counter()

    # Create unique seed for this simulation
    seed = BASE_SEED + sim_id + int(kappa * 100000) + sample_size + n_domains * 1000

    # Time data generation
    t0 = timing_module.perf_counter()
    generator = CovShiftDataGenerator(
        n_samples_per_domain=sample_size,
        n_domains=n_domains,
        concept_drift_degree=kappa,
        primary_lambda=PRIMARY_LAMBDA,
        primary_phi=PRIMARY_PHI,
        proxy_lambda=PROXY_LAMBDA,
        proxy_phi=PROXY_PHI,
        n_covariates=P,
        target_domain_means=mu_K,
        random_seed=seed,
    )
    data = generator.generate_data(include_cross_domain_weights=True) # NOTE This is important.

    # Create the PPI Estimator with domain splitting
    PPI_Estimator = CovShiftPPIEstimator(
        df=data,
        primary_outcome_column="primary_outcome",
        proxy_outcome_column="proxy_outcome",
        importance_weight_column="importance_weight",
        domain_column="domain",
        target_domain_value=n_domains,
    )

    # Create the Adjusted PPI Estimator
    Adjusted_Estimator = AdjustedCovShiftPPIEstimator(
        df=data,
        primary_outcome_column="primary_outcome",
        proxy_outcome_column="proxy_outcome",
        importance_weight_column="importance_weight",
        domain_column="domain",
        target_domain_value=n_domains,
        cross_domain_weight_pattern='weight_to_domain_{target_domain}'

    )
    time_data_generation = timing_module.perf_counter() - t0

    # Time primary estimate
    t0 = timing_module.perf_counter()
    primary_est = PPI_Estimator.compute_primary_mean_estimate(
        confidence_level=CONFIDENCE_LEVEL
    )
    time_primary_estimate = timing_module.perf_counter() - t0

    # Time proxy estimate
    t0 = timing_module.perf_counter()
    proxy_est = PPI_Estimator.compute_proxy_mean_estimate(
        confidence_level=CONFIDENCE_LEVEL
    )
    time_proxy_estimate = timing_module.perf_counter() - t0

    # Time PPI estimate
    t0 = timing_module.perf_counter()
    ppi_est = PPI_Estimator.compute_ppi_mean_estimate(
        weighted=False, confidence_level=CONFIDENCE_LEVEL
    )
    time_ppi_estimate = timing_module.perf_counter() - t0

    # Time PPI weighted estimate
    t0 = timing_module.perf_counter()
    ppi_weighted_est = PPI_Estimator.compute_ppi_mean_estimate(
        weighted=True, confidence_level=CONFIDENCE_LEVEL
    )
    time_ppi_weighted_estimate = timing_module.perf_counter() - t0

    # Time proxy + MoM adjustment estimate
    t0 = timing_module.perf_counter()
    proxy_adj_est = Adjusted_Estimator.compute_adjusted_proxy_estimate(
        confidence_level=CONFIDENCE_LEVEL, method="MoM"
    )
    time_proxy_adj_estimate = timing_module.perf_counter() - t0

    # Time PPI + MoM adjustment estimate
    t0 = timing_module.perf_counter()
    ppi_adj_est = Adjusted_Estimator.compute_adjusted_ppi_estimate(
        weighted=False, confidence_level=CONFIDENCE_LEVEL, method="MoM"
    )
    time_ppi_adj_estimate = timing_module.perf_counter() - t0

    # Time PPI weighted + MoM adjustment estimate
    t0 = timing_module.perf_counter()
    ppi_weighted_adj_est = Adjusted_Estimator.compute_adjusted_ppi_estimate(
        weighted=True, confidence_level=CONFIDENCE_LEVEL, method="MoM"
    )
    time_ppi_weighted_adj_estimate = timing_module.perf_counter() - t0

    # Time proxy + domain bootstrap estimate
    t0 = timing_module.perf_counter()
    proxy_domain_boot_est = Adjusted_Estimator.compute_domain_bootstrap_proxy_estimate(
        confidence_level=CONFIDENCE_LEVEL,
        n_bootstrap=N_BOOTSTRAP,
        seed=seed,
    )
    time_proxy_domain_boot_estimate = timing_module.perf_counter() - t0

    # Time PPI + domain bootstrap estimate
    t0 = timing_module.perf_counter()
    ppi_domain_boot_est = Adjusted_Estimator.compute_domain_bootstrap_ppi_estimate(
        weighted=False,
        confidence_level=CONFIDENCE_LEVEL,
        n_bootstrap=N_BOOTSTRAP,
        seed=seed,
    )
    time_ppi_domain_boot_estimate = timing_module.perf_counter() - t0

    # Time PPI weighted + domain bootstrap estimate
    t0 = timing_module.perf_counter()
    ppi_weighted_domain_boot_est = Adjusted_Estimator.compute_domain_bootstrap_ppi_estimate(
        weighted=True,
        confidence_level=CONFIDENCE_LEVEL,
        n_bootstrap=N_BOOTSTRAP,
        seed=seed,
    )
    time_ppi_weighted_domain_boot_estimate = timing_module.perf_counter() - t0

    time_total = timing_module.perf_counter() - total_start

    # Compute coverage for base estimates
    primary_covers = primary_est.lower_bound <= true_prevalence <= primary_est.upper_bound
    proxy_covers = proxy_est.lower_bound <= true_prevalence <= proxy_est.upper_bound
    ppi_covers = ppi_est.lower_bound <= true_prevalence <= ppi_est.upper_bound
    ppi_weighted_covers = ppi_weighted_est.lower_bound <= true_prevalence <= ppi_weighted_est.upper_bound

    # Compute coverage for MoM-adjusted estimates (adjusted values are now the primary fields)
    proxy_adj_covers = proxy_adj_est.lower_bound <= true_prevalence <= proxy_adj_est.upper_bound
    ppi_adj_covers = ppi_adj_est.lower_bound <= true_prevalence <= ppi_adj_est.upper_bound
    ppi_weighted_adj_covers = ppi_weighted_adj_est.lower_bound <= true_prevalence <= ppi_weighted_adj_est.upper_bound

    # Compute coverage for domain bootstrap estimates
    proxy_domain_boot_covers = proxy_domain_boot_est.domain_bootstrap_lower_bound <= true_prevalence <= proxy_domain_boot_est.domain_bootstrap_upper_bound
    ppi_domain_boot_covers = ppi_domain_boot_est.domain_bootstrap_lower_bound <= true_prevalence <= ppi_domain_boot_est.domain_bootstrap_upper_bound
    ppi_weighted_domain_boot_covers = ppi_weighted_domain_boot_est.domain_bootstrap_lower_bound <= true_prevalence <= ppi_weighted_domain_boot_est.domain_bootstrap_upper_bound

    return SimulationResult(
        kappa=kappa,
        sample_size=sample_size,
        n_domains=n_domains,
        sim_id=sim_id,
        true_prevalence=true_prevalence,
        # Primary estimate
        primary_estimate=primary_est.estimate_val,
        primary_lower=primary_est.lower_bound,
        primary_upper=primary_est.upper_bound,
        primary_covers=primary_covers,
        primary_ci_length=primary_est.upper_bound - primary_est.lower_bound,
        # Proxy estimate
        proxy_estimate=proxy_est.estimate_val,
        proxy_lower=proxy_est.lower_bound,
        proxy_upper=proxy_est.upper_bound,
        proxy_covers=proxy_covers,
        proxy_ci_length=proxy_est.upper_bound - proxy_est.lower_bound,
        # PPI estimate
        ppi_estimate=ppi_est.estimate_val,
        ppi_lower=ppi_est.lower_bound,
        ppi_upper=ppi_est.upper_bound,
        ppi_covers=ppi_covers,
        ppi_ci_length=ppi_est.upper_bound - ppi_est.lower_bound,
        # Weighted PPI estimate
        ppi_weighted_estimate=ppi_weighted_est.estimate_val,
        ppi_weighted_lower=ppi_weighted_est.lower_bound,
        ppi_weighted_upper=ppi_weighted_est.upper_bound,
        ppi_weighted_covers=ppi_weighted_covers,
        ppi_weighted_ci_length=ppi_weighted_est.upper_bound - ppi_weighted_est.lower_bound,
        # Proxy + MoM Adjustment
        proxy_adj_estimate=proxy_adj_est.estimate_val,
        proxy_adj_lower=proxy_adj_est.lower_bound,
        proxy_adj_upper=proxy_adj_est.upper_bound,
        proxy_adj_covers=proxy_adj_covers,
        proxy_adj_gamma_squared=proxy_adj_est.gamma_squared,
        proxy_adj_ci_length=proxy_adj_est.upper_bound - proxy_adj_est.lower_bound,
        # PPI + MoM Adjustment
        ppi_adj_estimate=ppi_adj_est.estimate_val,
        ppi_adj_lower=ppi_adj_est.lower_bound,
        ppi_adj_upper=ppi_adj_est.upper_bound,
        ppi_adj_covers=ppi_adj_covers,
        ppi_adj_gamma_squared=ppi_adj_est.gamma_squared,
        ppi_adj_ci_length=ppi_adj_est.upper_bound - ppi_adj_est.lower_bound,
        # PPI Weighted + MoM Adjustment
        ppi_weighted_adj_estimate=ppi_weighted_adj_est.estimate_val,
        ppi_weighted_adj_lower=ppi_weighted_adj_est.lower_bound,
        ppi_weighted_adj_upper=ppi_weighted_adj_est.upper_bound,
        ppi_weighted_adj_covers=ppi_weighted_adj_covers,
        ppi_weighted_adj_gamma_squared=ppi_weighted_adj_est.gamma_squared,
        ppi_weighted_adj_ci_length=ppi_weighted_adj_est.upper_bound - ppi_weighted_adj_est.lower_bound,
        # Proxy + Domain Bootstrap
        proxy_domain_boot_estimate=proxy_domain_boot_est.domain_bootstrap_point_estimate,
        proxy_domain_boot_lower=proxy_domain_boot_est.domain_bootstrap_lower_bound,
        proxy_domain_boot_upper=proxy_domain_boot_est.domain_bootstrap_upper_bound,
        proxy_domain_boot_covers=proxy_domain_boot_covers,
        proxy_domain_boot_ci_length=proxy_domain_boot_est.domain_bootstrap_upper_bound - proxy_domain_boot_est.domain_bootstrap_lower_bound,
        proxy_domain_boot_rho_mean=proxy_domain_boot_est.rho_mean,
        proxy_domain_boot_gamma_sq_mean=proxy_domain_boot_est.gamma_squared_mean,
        # PPI + Domain Bootstrap
        ppi_domain_boot_estimate=ppi_domain_boot_est.domain_bootstrap_point_estimate,
        ppi_domain_boot_lower=ppi_domain_boot_est.domain_bootstrap_lower_bound,
        ppi_domain_boot_upper=ppi_domain_boot_est.domain_bootstrap_upper_bound,
        ppi_domain_boot_covers=ppi_domain_boot_covers,
        ppi_domain_boot_ci_length=ppi_domain_boot_est.domain_bootstrap_upper_bound - ppi_domain_boot_est.domain_bootstrap_lower_bound,
        ppi_domain_boot_rho_mean=ppi_domain_boot_est.rho_mean,
        ppi_domain_boot_gamma_sq_mean=ppi_domain_boot_est.gamma_squared_mean,
        # PPI Weighted + Domain Bootstrap
        ppi_weighted_domain_boot_estimate=ppi_weighted_domain_boot_est.domain_bootstrap_point_estimate,
        ppi_weighted_domain_boot_lower=ppi_weighted_domain_boot_est.domain_bootstrap_lower_bound,
        ppi_weighted_domain_boot_upper=ppi_weighted_domain_boot_est.domain_bootstrap_upper_bound,
        ppi_weighted_domain_boot_covers=ppi_weighted_domain_boot_covers,
        ppi_weighted_domain_boot_ci_length=ppi_weighted_domain_boot_est.domain_bootstrap_upper_bound - ppi_weighted_domain_boot_est.domain_bootstrap_lower_bound,
        ppi_weighted_domain_boot_rho_mean=ppi_weighted_domain_boot_est.rho_mean,
        ppi_weighted_domain_boot_gamma_sq_mean=ppi_weighted_domain_boot_est.gamma_squared_mean,
        # Timing fields
        time_data_generation=time_data_generation,
        time_primary_estimate=time_primary_estimate,
        time_proxy_estimate=time_proxy_estimate,
        time_ppi_estimate=time_ppi_estimate,
        time_ppi_weighted_estimate=time_ppi_weighted_estimate,
        time_proxy_adj_estimate=time_proxy_adj_estimate,
        time_ppi_adj_estimate=time_ppi_adj_estimate,
        time_ppi_weighted_adj_estimate=time_ppi_weighted_adj_estimate,
        time_proxy_domain_boot_estimate=time_proxy_domain_boot_estimate,
        time_ppi_domain_boot_estimate=time_ppi_domain_boot_estimate,
        time_ppi_weighted_domain_boot_estimate=time_ppi_weighted_domain_boot_estimate,
        time_total=time_total,
    )

In [7]:
def run_simulation_batch(
    kappa: float,
    sample_size: int,
    n_domains: int,
    n_simulations: int,
    true_prevalence: float,
) -> List[SimulationResult]:
    """
    Run a batch of simulations sequentially.

    Args:
        kappa: Concept drift degree
        sample_size: Number of samples per domain
        n_domains: Number of domains (K)
        n_simulations: Number of simulation replications
        true_prevalence: True target prevalence

    Returns:
        List of SimulationResult objects
    """
    results = []

    for sim_id in range(n_simulations):
        try:
            result = run_single_simulation(kappa, sample_size, n_domains, sim_id, true_prevalence)
            results.append(result)
        except Exception as e:
            print(f"Simulation {sim_id} failed: {e}")

    return results


def compute_coverage_stats(
    results: List[SimulationResult],
) -> Dict[str, float]:
    """
    Compute coverage statistics, CI lengths, and timing from simulation results.

    Returns:
        Dictionary with coverage rates, average CI lengths, and timing for each method
    """
    n = len(results)
    if n == 0:
        return {
            # Base method coverage
            "primary_coverage": np.nan,
            "proxy_coverage": np.nan,
            "ppi_coverage": np.nan,
            "ppi_weighted_coverage": np.nan,
            # MoM-adjusted coverage
            "proxy_adj_coverage": np.nan,
            "ppi_adj_coverage": np.nan,
            "ppi_weighted_adj_coverage": np.nan,
            # Domain bootstrap coverage
            "proxy_domain_boot_coverage": np.nan,
            "ppi_domain_boot_coverage": np.nan,
            "ppi_weighted_domain_boot_coverage": np.nan,
            # Average gamma^2 estimates (MoM)
            "avg_proxy_adj_gamma_sq": np.nan,
            "avg_ppi_adj_gamma_sq": np.nan,
            "avg_ppi_weighted_adj_gamma_sq": np.nan,
            # Average domain bootstrap parameters
            "avg_proxy_domain_boot_rho": np.nan,
            "avg_proxy_domain_boot_gamma_sq": np.nan,
            "avg_ppi_domain_boot_rho": np.nan,
            "avg_ppi_domain_boot_gamma_sq": np.nan,
            "avg_ppi_weighted_domain_boot_rho": np.nan,
            "avg_ppi_weighted_domain_boot_gamma_sq": np.nan,
            # Average CI lengths (base methods)
            "avg_primary_ci_length": np.nan,
            "avg_proxy_ci_length": np.nan,
            "avg_ppi_ci_length": np.nan,
            "avg_ppi_weighted_ci_length": np.nan,
            # Average CI lengths (MoM-adjusted)
            "avg_proxy_adj_ci_length": np.nan,
            "avg_ppi_adj_ci_length": np.nan,
            "avg_ppi_weighted_adj_ci_length": np.nan,
            # Average CI lengths (Domain Bootstrap)
            "avg_proxy_domain_boot_ci_length": np.nan,
            "avg_ppi_domain_boot_ci_length": np.nan,
            "avg_ppi_weighted_domain_boot_ci_length": np.nan,
            # Timing statistics (average time in seconds)
            "avg_time_data_generation": np.nan,
            "avg_time_primary_estimate": np.nan,
            "avg_time_proxy_estimate": np.nan,
            "avg_time_ppi_estimate": np.nan,
            "avg_time_ppi_weighted_estimate": np.nan,
            "avg_time_proxy_adj_estimate": np.nan,
            "avg_time_ppi_adj_estimate": np.nan,
            "avg_time_ppi_weighted_adj_estimate": np.nan,
            "avg_time_proxy_domain_boot_estimate": np.nan,
            "avg_time_ppi_domain_boot_estimate": np.nan,
            "avg_time_ppi_weighted_domain_boot_estimate": np.nan,
            "avg_time_total": np.nan,
        }

    return {
        # Base method coverage
        "primary_coverage": sum(r.primary_covers for r in results) / n,
        "proxy_coverage": sum(r.proxy_covers for r in results) / n,
        "ppi_coverage": sum(r.ppi_covers for r in results) / n,
        "ppi_weighted_coverage": sum(r.ppi_weighted_covers for r in results) / n,
        # MoM-adjusted coverage
        "proxy_adj_coverage": sum(r.proxy_adj_covers for r in results) / n,
        "ppi_adj_coverage": sum(r.ppi_adj_covers for r in results) / n,
        "ppi_weighted_adj_coverage": sum(r.ppi_weighted_adj_covers for r in results) / n,
        # Domain bootstrap coverage
        "proxy_domain_boot_coverage": sum(r.proxy_domain_boot_covers for r in results) / n,
        "ppi_domain_boot_coverage": sum(r.ppi_domain_boot_covers for r in results) / n,
        "ppi_weighted_domain_boot_coverage": sum(r.ppi_weighted_domain_boot_covers for r in results) / n,
        # Average gamma^2 estimates (MoM)
        "avg_proxy_adj_gamma_sq": np.mean([r.proxy_adj_gamma_squared for r in results]),
        "avg_ppi_adj_gamma_sq": np.mean([r.ppi_adj_gamma_squared for r in results]),
        "avg_ppi_weighted_adj_gamma_sq": np.mean([r.ppi_weighted_adj_gamma_squared for r in results]),
        # Average domain bootstrap parameters
        "avg_proxy_domain_boot_rho": np.mean([r.proxy_domain_boot_rho_mean for r in results]),
        "avg_proxy_domain_boot_gamma_sq": np.mean([r.proxy_domain_boot_gamma_sq_mean for r in results]),
        "avg_ppi_domain_boot_rho": np.mean([r.ppi_domain_boot_rho_mean for r in results]),
        "avg_ppi_domain_boot_gamma_sq": np.mean([r.ppi_domain_boot_gamma_sq_mean for r in results]),
        "avg_ppi_weighted_domain_boot_rho": np.mean([r.ppi_weighted_domain_boot_rho_mean for r in results]),
        "avg_ppi_weighted_domain_boot_gamma_sq": np.mean([r.ppi_weighted_domain_boot_gamma_sq_mean for r in results]),
        # Average CI lengths (base methods)
        "avg_primary_ci_length": np.mean([r.primary_ci_length for r in results]),
        "avg_proxy_ci_length": np.mean([r.proxy_ci_length for r in results]),
        "avg_ppi_ci_length": np.mean([r.ppi_ci_length for r in results]),
        "avg_ppi_weighted_ci_length": np.mean([r.ppi_weighted_ci_length for r in results]),
        # Average CI lengths (MoM-adjusted)
        "avg_proxy_adj_ci_length": np.mean([r.proxy_adj_ci_length for r in results]),
        "avg_ppi_adj_ci_length": np.mean([r.ppi_adj_ci_length for r in results]),
        "avg_ppi_weighted_adj_ci_length": np.mean([r.ppi_weighted_adj_ci_length for r in results]),
        # Average CI lengths (Domain Bootstrap)
        "avg_proxy_domain_boot_ci_length": np.mean([r.proxy_domain_boot_ci_length for r in results]),
        "avg_ppi_domain_boot_ci_length": np.mean([r.ppi_domain_boot_ci_length for r in results]),
        "avg_ppi_weighted_domain_boot_ci_length": np.mean([r.ppi_weighted_domain_boot_ci_length for r in results]),
        # Timing statistics (average time in seconds)
        "avg_time_data_generation": np.mean([r.time_data_generation for r in results]),
        "avg_time_primary_estimate": np.mean([r.time_primary_estimate for r in results]),
        "avg_time_proxy_estimate": np.mean([r.time_proxy_estimate for r in results]),
        "avg_time_ppi_estimate": np.mean([r.time_ppi_estimate for r in results]),
        "avg_time_ppi_weighted_estimate": np.mean([r.time_ppi_weighted_estimate for r in results]),
        "avg_time_proxy_adj_estimate": np.mean([r.time_proxy_adj_estimate for r in results]),
        "avg_time_ppi_adj_estimate": np.mean([r.time_ppi_adj_estimate for r in results]),
        "avg_time_ppi_weighted_adj_estimate": np.mean([r.time_ppi_weighted_adj_estimate for r in results]),
        "avg_time_proxy_domain_boot_estimate": np.mean([r.time_proxy_domain_boot_estimate for r in results]),
        "avg_time_ppi_domain_boot_estimate": np.mean([r.time_ppi_domain_boot_estimate for r in results]),
        "avg_time_ppi_weighted_domain_boot_estimate": np.mean([r.time_ppi_weighted_domain_boot_estimate for r in results]),
        "avg_time_total": np.mean([r.time_total for r in results]),
    }

In [ ]:
# =============================================================================
# RUN ALL SIMULATIONS
# =============================================================================

all_results = []  # Store all individual simulation results
coverage_summary = []  # Store aggregated coverage statistics

total_combinations = len(N_DOMAINS_VALUES) * len(KAPPA_VALUES) * len(SAMPLE_SIZES)
current_combination = 0

print(f"Running {total_combinations} combinations x {N_SIMULATIONS} simulations = {total_combinations * N_SIMULATIONS:,} total simulations")
print(f"  - N_DOMAINS_VALUES: {N_DOMAINS_VALUES}")
print(f"  - KAPPA_VALUES: {KAPPA_VALUES}")
print(f"  - SAMPLE_SIZES: {list(SAMPLE_SIZES)}")
print("="*80)

start_time_total = time.time()

for n_domains in N_DOMAINS_VALUES:
    print(f"\n{'='*80}")
    print(f"Running simulations for K={n_domains} domains")
    print(f"{'='*80}")

    for kappa in KAPPA_VALUES:
        true_prev = TRUE_PREVALENCES[kappa]

        for sample_size in SAMPLE_SIZES:
            current_combination += 1
            print(f"\n[{current_combination}/{total_combinations}] K={n_domains}, kappa={kappa}, n={sample_size}")

            start_time = time.time()

            # Run simulations in parallel
            results = run_simulation_batch(
                kappa=kappa,
                sample_size=sample_size,
                n_domains=n_domains,
                n_simulations=N_SIMULATIONS,
                true_prevalence=true_prev,
            )

            elapsed = time.time() - start_time

            # Store individual results
            all_results.extend(results)

            # Compute coverage statistics
            coverage = compute_coverage_stats(results)
            coverage_summary.append({
                "n_domains": n_domains,
                "kappa": kappa,
                "sample_size": sample_size,
                "true_prevalence": true_prev,
                "n_simulations": len(results),
                **coverage,
            })

            print(f"  Completed in {elapsed:.1f}s (avg {coverage['avg_time_total']*1000:.1f}ms per sim)")
            print(f"  Base Coverage:     Primary={coverage['primary_coverage']:.3f}, "
                  f"Proxy={coverage['proxy_coverage']:.3f}, "
                  f"PPI={coverage['ppi_coverage']:.3f}, "
                  f"PPI-W={coverage['ppi_weighted_coverage']:.3f}")
            print(f"  MoM-Adjusted:      Proxy+Adj={coverage['proxy_adj_coverage']:.3f}, "
                  f"PPI+Adj={coverage['ppi_adj_coverage']:.3f}, "
                  f"PPI-W+Adj={coverage['ppi_weighted_adj_coverage']:.3f}")
            print(f"  Domain Bootstrap:  Proxy+DomBoot={coverage['proxy_domain_boot_coverage']:.3f}, "
                  f"PPI+DomBoot={coverage['ppi_domain_boot_coverage']:.3f}, "
                  f"PPI-W+DomBoot={coverage['ppi_weighted_domain_boot_coverage']:.3f}")

            # Print timing breakdown
            print(f"  Timing (avg ms):   DataGen={coverage['avg_time_data_generation']*1000:.1f}, "
                  f"Primary={coverage['avg_time_primary_estimate']*1000:.2f}, "
                  f"Proxy={coverage['avg_time_proxy_estimate']*1000:.2f}, "
                  f"PPI={coverage['avg_time_ppi_estimate']*1000:.2f}")
            print(f"                     ProxyAdj={coverage['avg_time_proxy_adj_estimate']*1000:.2f}, "
                  f"PPIAdj={coverage['avg_time_ppi_adj_estimate']*1000:.2f}, "
                  f"PPI-WAdj={coverage['avg_time_ppi_weighted_adj_estimate']*1000:.2f}")
            print(f"                     ProxyBoot={coverage['avg_time_proxy_domain_boot_estimate']*1000:.1f}, "
                  f"PPIBoot={coverage['avg_time_ppi_domain_boot_estimate']*1000:.1f}, "
                  f"PPI-WBoot={coverage['avg_time_ppi_weighted_domain_boot_estimate']*1000:.1f}")

total_elapsed = time.time() - start_time_total
print(f"\n{'='*80}")
print(f"Total simulation time: {total_elapsed/60:.1f} minutes")

Running 18 combinations x 3000 simulations = 54,000 total simulations
  - N_DOMAINS_VALUES: [5, 10, 25]
  - KAPPA_VALUES: [0, 1, 5]
  - SAMPLE_SIZES: [np.int64(500), np.int64(1000)]

Running simulations for K=5 domains

[1/18] K=5, kappa=0, n=500


In [ ]:
# =============================================================================
# SAVE RESULTS TO CSV
# =============================================================================

# Convert results to DataFrames
df_all_results = pd.DataFrame([vars(r) for r in all_results])
df_coverage_summary = pd.DataFrame(coverage_summary)

# Save individual simulation results
results_path = OUTPUT_DIR / "simulation_results_all.csv"
df_all_results.to_csv(results_path, index=False)
print(f"Saved {len(df_all_results):,} individual results to: {results_path}")

# Save coverage summary
coverage_path = OUTPUT_DIR / "coverage_summary.csv"
df_coverage_summary.to_csv(coverage_path, index=False)
print(f"Saved coverage summary to: {coverage_path}")

# Save true prevalences
prevalence_path = OUTPUT_DIR / "true_prevalences.csv"
pd.DataFrame([
    {"kappa": k, "true_prevalence": v, "n_mc_samples": N_MC_SAMPLES}
    for k, v in TRUE_PREVALENCES.items()
]).to_csv(prevalence_path, index=False)
print(f"Saved true prevalences to: {prevalence_path}")

# Display summary
print("\nCoverage Summary:")
df_coverage_summary

In [ ]:
# =============================================================================
# LOAD RESULTS FROM CSV
# =============================================================================

# Load individual simulation results
results_path = OUTPUT_DIR / "simulation_results_all.csv"
df_all_results = pd.read_csv(results_path)
print(f"Loaded {len(df_all_results):,} individual results from: {results_path}")

# Load coverage summary
coverage_path = OUTPUT_DIR / "coverage_summary.csv"
df_coverage_summary = pd.read_csv(coverage_path)
print(f"Loaded coverage summary from: {coverage_path}")

# Load true prevalences
prevalence_path = OUTPUT_DIR / "true_prevalences.csv"
df_true_prevalences = pd.read_csv(prevalence_path)
print(f"Loaded true prevalences from: {prevalence_path}")

# Display summary
print("\nCoverage Summary:")
df_coverage_summary

In [ ]:
# =============================================================================
# COVERAGE PLOTS
# =============================================================================
# Standardized color scheme:
#   - Primary: Blue (#1f77b4)
#   - Proxy: Red (#d62728)
#   - PPI: Orange (#ff7f0e)
#   - PPI-W: Purple (#9467bd)
# Standardized line/marker scheme:
#   - Base methods: solid line (-), circle marker (o)
#   - Adjusted (+Adj): dashed line (--), square marker (s)
#   - Bootstrap (+Boot): dotted line (:), diamond marker (D)

# Color constants
COLOR_PRIMARY = "#1f77b4"  # Blue
COLOR_PROXY = "#d62728"    # Red
COLOR_PPI = "#ff7f0e"      # Orange
COLOR_PPI_W = "#9467bd"    # Purple

def plot_coverage_by_sample_size(df: pd.DataFrame, save_dir: Path = OUTPUT_DIR):
    """
    Create coverage vs sample size plots for each (n_domains, kappa) combination.

    Each plot shows the coverage of all methods (base, MoM-adjusted, Domain Bootstrap)
    as a function of sample size.
    """
    methods = [
        # Base methods - solid line, circle marker
        ("primary_coverage", "Primary Only", COLOR_PRIMARY, "o", "-"),
        ("proxy_coverage", "Proxy Only", COLOR_PROXY, "o", "-"),
        ("ppi_coverage", "PPI", COLOR_PPI, "o", "-"),
        ("ppi_weighted_coverage", "PPI-W", COLOR_PPI_W, "o", "-"),
        # MoM-adjusted methods - dashed line, square marker
        ("proxy_adj_coverage", "Proxy+Adj", COLOR_PROXY, "s", "--"),
        ("ppi_adj_coverage", "PPI+Adj", COLOR_PPI, "s", "--"),
        ("ppi_weighted_adj_coverage", "PPI-W+Adj", COLOR_PPI_W, "s", "--"),
        # Domain bootstrap methods - dotted line, diamond marker
        ("proxy_domain_boot_coverage", "Proxy+Boot", COLOR_PROXY, "D", ":"),
        ("ppi_domain_boot_coverage", "PPI+Boot", COLOR_PPI, "D", ":"),
        ("ppi_weighted_domain_boot_coverage", "PPI-W+Boot", COLOR_PPI_W, "D", ":"),
    ]

    # Create individual plots for each (n_domains, kappa) combination
    for n_domains in N_DOMAINS_VALUES:
        for kappa in KAPPA_VALUES:
            df_subset = df[(df["n_domains"] == n_domains) & (df["kappa"] == kappa)].sort_values("sample_size")

            fig, ax = plt.subplots(figsize=(14, 8))

            for col, label, color, marker, linestyle in methods:
                ax.plot(
                    df_subset["sample_size"],
                    df_subset[col],
                    label=label,
                    color=color,
                    marker=marker,
                    markersize=8,
                    linewidth=2,
                    linestyle=linestyle,
                )

            # Add reference line at 95%
            ax.axhline(y=0.95, color="gray", linestyle=":", linewidth=1.5, label="Nominal (95%)")

            ax.set_xlabel("Sample Size per Domain", fontsize=12)
            ax.set_ylabel("Coverage Rate", fontsize=12)
            ax.set_title(f"95% CI Coverage vs Sample Size (K={n_domains} domains, κ={kappa})", fontsize=14)
            ax.set_xscale("log")
            ax.set_ylim(0, 1.05)
            ax.legend(loc="lower right", fontsize=8, ncol=2)
            ax.grid(True, alpha=0.3)

            plt.tight_layout()

            # Save figure
            fig_path = save_dir / f"coverage_vs_sample_size_K{n_domains}_kappa_{kappa}.png"
            plt.savefig(fig_path, dpi=150, bbox_inches="tight")
            print(f"Saved plot: {fig_path}")

            plt.show()


def plot_coverage_by_n_domains(df: pd.DataFrame, save_dir: Path = OUTPUT_DIR):
    """
    Create coverage vs number of domains plots for each (kappa, sample_size) combination.

    Shows how coverage changes as the number of source domains increases.
    """
    methods = [
        # MoM-adjusted methods - dashed line, square marker
        ("proxy_adj_coverage", "Proxy+Adj", COLOR_PROXY, "s", "--"),
        ("ppi_adj_coverage", "PPI+Adj", COLOR_PPI, "s", "--"),
        ("ppi_weighted_adj_coverage", "PPI-W+Adj", COLOR_PPI_W, "s", "--"),
        # Domain bootstrap methods - dotted line, diamond marker
        ("proxy_domain_boot_coverage", "Proxy+Boot", COLOR_PROXY, "D", ":"),
        ("ppi_domain_boot_coverage", "PPI+Boot", COLOR_PPI, "D", ":"),
        ("ppi_weighted_domain_boot_coverage", "PPI-W+Boot", COLOR_PPI_W, "D", ":"),
    ]

    # Create a grid of plots: rows = kappa, cols = sample_size (subset)
    sample_sizes_subset = [SAMPLE_SIZES[0], SAMPLE_SIZES[len(SAMPLE_SIZES)//2], SAMPLE_SIZES[-1]]

    n_rows = len(KAPPA_VALUES)
    n_cols = len(sample_sizes_subset)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows), squeeze=False)

    for i, kappa in enumerate(KAPPA_VALUES):
        for j, sample_size in enumerate(sample_sizes_subset):
            ax = axes[i, j]
            df_subset = df[(df["kappa"] == kappa) & (df["sample_size"] == sample_size)].sort_values("n_domains")

            for col, label, color, marker, linestyle in methods:
                ax.plot(
                    df_subset["n_domains"],
                    df_subset[col],
                    label=label,
                    color=color,
                    marker=marker,
                    markersize=8,
                    linewidth=2,
                    linestyle=linestyle,
                )

            ax.axhline(y=0.95, color="gray", linestyle=":", linewidth=1.5)
            ax.set_xlabel("Number of Domains (K)", fontsize=10)
            ax.set_ylabel("Coverage Rate", fontsize=10)
            ax.set_title(f"κ={kappa}, n={sample_size}", fontsize=11)
            ax.set_ylim(0.7, 1.05)
            ax.set_xticks(N_DOMAINS_VALUES)
            ax.grid(True, alpha=0.3)

    # Add legend to the figure
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=9, bbox_to_anchor=(0.5, -0.05))

    fig.suptitle("Coverage vs Number of Domains (Adjusted & Domain Bootstrap Methods)", fontsize=14, y=1.02)
    plt.tight_layout()

    # Save figure
    fig_path = save_dir / "coverage_vs_n_domains.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"Saved plot: {fig_path}")

    plt.show()


# Generate plots
plot_coverage_by_sample_size(df_coverage_summary)
plot_coverage_by_n_domains(df_coverage_summary)


In [ ]:
# =============================================================================
# COMBINED PLOTS - All kappa values with n_domains as line colors
# =============================================================================
# Standardized color scheme:
#   - Primary: Blue (#1f77b4)
#   - Proxy: Red (#d62728)
#   - PPI: Orange (#ff7f0e)
#   - PPI-W: Purple (#9467bd)
# Standardized line/marker scheme:
#   - Base methods: solid line (-), circle marker (o)
#   - Adjusted (+Adj): dashed line (--), square marker (s)
#   - Bootstrap (+Boot): dotted line (:), diamond marker (D)

# Color constants (same as plot_coverage)
COLOR_PRIMARY = "#1f77b4"  # Blue
COLOR_PROXY = "#d62728"    # Red
COLOR_PPI = "#ff7f0e"      # Orange
COLOR_PPI_W = "#9467bd"    # Purple

def plot_coverage_combined(df: pd.DataFrame, save_dir: Path = OUTPUT_DIR):
    """
    Create a combined figure with all kappa values as subplots.
    Shows how coverage varies with sample size, with n_domains as different lines.
    """
    # Focus on key methods for cleaner visualization
    methods = [
        ("ppi_adj_coverage", "PPI+Adj", COLOR_PPI, "s", "--"),
        ("ppi_weighted_adj_coverage", "PPI-W+Adj", COLOR_PPI_W, "s", "--"),
        ("ppi_domain_boot_coverage", "PPI+Boot", COLOR_PPI, "D", ":"),
        ("ppi_weighted_domain_boot_coverage", "PPI-W+Boot", COLOR_PPI_W, "D", ":"),
    ]

    colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(N_DOMAINS_VALUES)))

    n_kappas = len(KAPPA_VALUES)
    n_methods = len(methods)

    fig, axes = plt.subplots(n_kappas, n_methods, figsize=(5*n_methods, 4*n_kappas), squeeze=False)

    for i, kappa in enumerate(KAPPA_VALUES):
        for j, (col, method_name, base_color, marker, linestyle) in enumerate(methods):
            ax = axes[i, j]

            for k, n_domains in enumerate(N_DOMAINS_VALUES):
                df_subset = df[(df["kappa"] == kappa) & (df["n_domains"] == n_domains)].sort_values("sample_size")
                ax.plot(
                    df_subset["sample_size"],
                    df_subset[col],
                    label=f"K={n_domains}",
                    color=colors[k],
                    marker=marker,
                    markersize=4,
                    linewidth=1.5,
                    linestyle=linestyle,
                )

            ax.axhline(y=0.95, color="gray", linestyle=":", linewidth=1)
            ax.set_xlabel("Sample Size", fontsize=9)
            ax.set_ylabel("Coverage", fontsize=9)
            ax.set_title(f"κ={kappa}: {method_name}", fontsize=10)
            ax.set_xscale("log")
            ax.set_ylim(0.7, 1.05)
            ax.grid(True, alpha=0.3)

    # Add legend to the figure
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=len(N_DOMAINS_VALUES), fontsize=9, bbox_to_anchor=(0.5, -0.02))

    fig.suptitle("95% CI Coverage: Adjusted & Domain Bootstrap Methods by Number of Domains", fontsize=14, y=1.01)
    plt.tight_layout()

    # Save combined figure
    fig_path = save_dir / "coverage_combined_by_n_domains.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"Saved combined plot: {fig_path}")

    plt.show()


def plot_coverage_grid_by_n_domains(df: pd.DataFrame, save_dir: Path = OUTPUT_DIR):
    """
    Create a grid where rows are n_domains and columns are kappa values.
    Shows all methods for each (n_domains, kappa) combination.
    """
    methods = [
        # MoM-adjusted methods - dashed line, square marker
        ("proxy_adj_coverage", "Proxy+Adj", COLOR_PROXY, "s", "--"),
        ("ppi_adj_coverage", "PPI+Adj", COLOR_PPI, "s", "--"),
        ("ppi_weighted_adj_coverage", "PPI-W+Adj", COLOR_PPI_W, "s", "--"),
        # Domain bootstrap methods - dotted line, diamond marker
        ("proxy_domain_boot_coverage", "Proxy+Boot", COLOR_PROXY, "D", ":"),
        ("ppi_domain_boot_coverage", "PPI+Boot", COLOR_PPI, "D", ":"),
        ("ppi_weighted_domain_boot_coverage", "PPI-W+Boot", COLOR_PPI_W, "D", ":"),
    ]

    n_rows = len(N_DOMAINS_VALUES)
    n_cols = len(KAPPA_VALUES)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 3.5*n_rows), squeeze=False)

    for i, n_domains in enumerate(N_DOMAINS_VALUES):
        for j, kappa in enumerate(KAPPA_VALUES):
            ax = axes[i, j]
            df_subset = df[(df["n_domains"] == n_domains) & (df["kappa"] == kappa)].sort_values("sample_size")

            for col, label, color, marker, linestyle in methods:
                ax.plot(
                    df_subset["sample_size"],
                    df_subset[col],
                    label=label,
                    color=color,
                    marker=marker,
                    markersize=4,
                    linewidth=1.5,
                    linestyle=linestyle,
                )

            ax.axhline(y=0.95, color="gray", linestyle=":", linewidth=1)
            ax.set_xlabel("Sample Size", fontsize=8)
            ax.set_ylabel("Coverage", fontsize=8)
            ax.set_title(f"K={n_domains}, κ={kappa}", fontsize=9)
            ax.set_xscale("log")
            ax.set_ylim(0.7, 1.05)
            ax.grid(True, alpha=0.3)

    # Add legend to the figure
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=8, bbox_to_anchor=(0.5, -0.04))

    fig.suptitle("95% CI Coverage Grid: Rows=Domains, Columns=Kappa", fontsize=14, y=1.01)
    plt.tight_layout()

    # Save figure
    fig_path = save_dir / "coverage_grid_n_domains_kappa.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"Saved grid plot: {fig_path}")

    plt.show()


# Generate combined plots
plot_coverage_combined(df_coverage_summary)
plot_coverage_grid_by_n_domains(df_coverage_summary)


In [ ]:
# =============================================================================
# METHOD-CENTRIC PLOTS - One plot per method across all kappas and n_domains
# =============================================================================
# Standardized color scheme:
#   - Primary: Blue (#1f77b4)
#   - Proxy: Red (#d62728)
#   - PPI: Orange (#ff7f0e)
#   - PPI-W: Purple (#9467bd)
# Standardized line/marker scheme:
#   - Base methods: solid line (-), circle marker (o)
#   - Adjusted (+Adj): dashed line (--), square marker (s)
#   - Bootstrap (+Boot): dotted line (:), diamond marker (D)

# Color constants
COLOR_PRIMARY = "#1f77b4"  # Blue
COLOR_PROXY = "#d62728"    # Red
COLOR_PPI = "#ff7f0e"      # Orange
COLOR_PPI_W = "#9467bd"    # Purple

def plot_coverage_by_method(df: pd.DataFrame, save_dir: Path = OUTPUT_DIR):
    """
    Create one plot per method showing how coverage varies with kappa.
    Uses n_domains as different line styles.
    """
    methods = [
        ("proxy_coverage", "Proxy Only", COLOR_PROXY, "o", "-"),
        ("ppi_coverage", "PPI", COLOR_PPI, "o", "-"),
        ("ppi_weighted_coverage", "PPI-W", COLOR_PPI_W, "o", "-"),
        ("proxy_adj_coverage", "Proxy+Adj", COLOR_PROXY, "s", "--"),
        ("ppi_adj_coverage", "PPI+Adj", COLOR_PPI, "s", "--"),
        ("ppi_weighted_adj_coverage", "PPI-W+Adj", COLOR_PPI_W, "s", "--"),
        ("proxy_domain_boot_coverage", "Proxy+Boot", COLOR_PROXY, "D", ":"),
        ("ppi_domain_boot_coverage", "PPI+Boot", COLOR_PPI, "D", ":"),
        ("ppi_weighted_domain_boot_coverage", "PPI-W+Boot", COLOR_PPI_W, "D", ":"),
    ]

    kappa_colors = plt.cm.viridis(np.linspace(0, 0.9, len(KAPPA_VALUES)))

    n_methods = len(methods)
    n_cols = 3
    n_rows = (n_methods + n_cols - 1) // n_cols

    # Create one figure per n_domains value
    for n_domains_idx, n_domains in enumerate(N_DOMAINS_VALUES):
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
        axes = axes.flatten()

        for idx, (col, title, base_color, marker, linestyle) in enumerate(methods):
            ax = axes[idx]

            for i, kappa in enumerate(KAPPA_VALUES):
                df_subset = df[(df["kappa"] == kappa) & (df["n_domains"] == n_domains)].sort_values("sample_size")
                ax.plot(
                    df_subset["sample_size"],
                    df_subset[col],
                    label=f"κ={kappa}",
                    color=kappa_colors[i],
                    marker=marker,
                    markersize=5,
                    linewidth=1.5,
                    linestyle=linestyle,
                )

            ax.axhline(y=0.95, color="gray", linestyle="--", linewidth=1, label="Nominal (95%)")
            ax.set_xlabel("Sample Size per Domain", fontsize=11)
            ax.set_ylabel("Coverage Rate", fontsize=11)
            ax.set_title(title, fontsize=13, color=base_color)
            ax.set_xscale("log")
            ax.set_ylim(0.0, 1.05)
            ax.legend(loc="lower right", fontsize=8)
            ax.grid(True, alpha=0.3)

        # Hide unused subplots
        for idx in range(len(methods), len(axes)):
            axes[idx].set_visible(False)

        fig.suptitle(f"Coverage by Method (K={n_domains} domains)", fontsize=14, y=1.01)
        plt.tight_layout()

        fig_path = save_dir / f"coverage_by_method_K{n_domains}.png"
        plt.savefig(fig_path, dpi=150, bbox_inches="tight")
        print(f"Saved method comparison plot: {fig_path}")

        plt.show()


def plot_base_vs_adjusted(df: pd.DataFrame, save_dir: Path = OUTPUT_DIR):
    """
    Create comparison plots showing base methods vs their MoM and Domain Bootstrap adjusted versions.
    One plot per n_domains value.
    """
    for n_domains in N_DOMAINS_VALUES:
        comparisons = [
            ("ppi_coverage", "ppi_adj_coverage", "ppi_domain_boot_coverage", "PPI", COLOR_PPI),
            ("ppi_weighted_coverage", "ppi_weighted_adj_coverage", "ppi_weighted_domain_boot_coverage", "PPI-W", COLOR_PPI_W),
        ]

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        for idx, (base_col, adj_col, boot_col, title, color) in enumerate(comparisons):
            ax = axes[idx]

            for kappa in KAPPA_VALUES:
                df_subset = df[(df["kappa"] == kappa) & (df["n_domains"] == n_domains)].sort_values("sample_size")

                # Base method - solid line, circle marker
                ax.plot(
                    df_subset["sample_size"],
                    df_subset[base_col],
                    color=color,
                    linestyle="-",
                    marker="o",
                    markersize=4,
                    linewidth=1.5,
                    alpha=0.4,
                    label=f"Base (κ={kappa})" if kappa == KAPPA_VALUES[0] else None,
                )
                # MoM-adjusted - dashed line, square marker
                ax.plot(
                    df_subset["sample_size"],
                    df_subset[adj_col],
                    color=color,
                    linestyle="--",
                    marker="s",
                    markersize=4,
                    linewidth=2,
                    label=f"Adj (κ={kappa})" if kappa == KAPPA_VALUES[0] else None,
                )
                # Domain Bootstrap - dotted line, diamond marker
                ax.plot(
                    df_subset["sample_size"],
                    df_subset[boot_col],
                    color=color,
                    linestyle=":",
                    marker="D",
                    markersize=4,
                    linewidth=1.5,
                    alpha=0.7,
                    label=f"Boot (κ={kappa})" if kappa == KAPPA_VALUES[0] else None,
                )

            ax.axhline(y=0.95, color="gray", linestyle=":", linewidth=1.5)
            ax.set_xlabel("Sample Size per Domain", fontsize=11)
            ax.set_ylabel("Coverage Rate", fontsize=11)
            ax.set_title(f"{title}: Base vs Adj vs Boot", fontsize=13, color=color)
            ax.set_xscale("log")
            ax.set_ylim(0.0, 1.05)
            ax.grid(True, alpha=0.3)
            ax.legend(loc="lower right", fontsize=9)

        fig.suptitle(f"Coverage Comparison (K={n_domains} domains)", fontsize=14, y=1.02)
        plt.tight_layout()

        fig_path = save_dir / f"coverage_base_vs_adjusted_K{n_domains}.png"
        plt.savefig(fig_path, dpi=150, bbox_inches="tight")
        print(f"Saved base vs adjusted comparison plot: {fig_path}")

        plt.show()


def plot_adj_vs_domboot_by_n_domains(df: pd.DataFrame, save_dir: Path = OUTPUT_DIR):
    """
    Compare MoM adjustment vs Domain Bootstrap methods across different n_domains.
    Shows how the gap between methods changes with the number of source domains.
    """
    methods = [
        ("ppi_adj_coverage", "ppi_domain_boot_coverage", "PPI", COLOR_PPI),
        ("ppi_weighted_adj_coverage", "ppi_weighted_domain_boot_coverage", "PPI-W", COLOR_PPI_W),
    ]

    n_rows = len(methods)
    n_cols = len(KAPPA_VALUES)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 4*n_rows), squeeze=False)

    n_domain_colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(N_DOMAINS_VALUES)))

    for i, (adj_col, boot_col, method_name, base_color) in enumerate(methods):
        for j, kappa in enumerate(KAPPA_VALUES):
            ax = axes[i, j]

            for k, n_domains in enumerate(N_DOMAINS_VALUES):
                df_subset = df[(df["kappa"] == kappa) & (df["n_domains"] == n_domains)].sort_values("sample_size")

                # MoM adjustment - dashed line, square marker
                ax.plot(
                    df_subset["sample_size"],
                    df_subset[adj_col],
                    color=n_domain_colors[k],
                    linestyle="--",
                    marker="s",
                    markersize=5,
                    linewidth=2,
                    label=f"Adj K={n_domains}" if j == 0 else None,
                )
                # Domain Bootstrap - dotted line, diamond marker
                ax.plot(
                    df_subset["sample_size"],
                    df_subset[boot_col],
                    color=n_domain_colors[k],
                    linestyle=":",
                    marker="D",
                    markersize=5,
                    linewidth=2,
                    alpha=0.7,
                    label=f"Boot K={n_domains}" if j == 0 else None,
                )

            ax.axhline(y=0.95, color="gray", linestyle=":", linewidth=1.5)
            ax.set_xlabel("Sample Size", fontsize=10)
            ax.set_ylabel("Coverage", fontsize=10)
            ax.set_title(f"{method_name}: κ={kappa}", fontsize=11, color=base_color)
            ax.set_xscale("log")
            ax.set_ylim(0.0, 1.05)
            ax.grid(True, alpha=0.3)

    # Add legend to first row
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.04))

    fig.suptitle("MoM Adjustment vs Domain Bootstrap: Effect of Number of Domains", fontsize=14, y=1.01)
    plt.tight_layout()

    fig_path = save_dir / "adj_vs_domboot_by_n_domains.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"Saved adjustment comparison plot: {fig_path}")

    plt.show()


# Generate method-centric plots
plot_coverage_by_method(df_coverage_summary)

# Generate base vs adjusted comparison
plot_base_vs_adjusted(df_coverage_summary)

# Generate adjustment vs domain bootstrap comparison
plot_adj_vs_domboot_by_n_domains(df_coverage_summary)

In [ ]:
# =============================================================================
# Publication Version: METHOD-CENTRIC PLOTS - One plot per method across all kappas and n_domains
# =============================================================================
# Standardized color scheme:
#   - Primary: Blue (#1f77b4)
#   - Proxy: Red (#d62728)
#   - PPI: Orange (#ff7f0e)
#   - PPI-W: Purple (#9467bd)
# Standardized line/marker scheme:
#   - Base methods: solid line (-), circle marker (o)
#   - Adjusted (+Adj): dashed line (--), square marker (s)
#   - Bootstrap (+Boot): dotted line (:), diamond marker (D)

# Color constants
COLOR_PRIMARY = "#1f77b4"  # Blue
COLOR_PROXY = "#d62728"  # Red
COLOR_PPI = "#ff7f0e"  # Orange
COLOR_PPI_W = "#9467bd"  # Purple


def plot_coverage_by_method(df: pd.DataFrame, save_dir: Path = OUTPUT_DIR):
    """
    Create one plot per method showing how coverage varies with kappa.
    Uses n_domains as different line styles.
    """
    methods = [
        ("proxy_coverage", "Proxy Only", COLOR_PROXY, "o", "-"),
        ("ppi_coverage", "PPI", COLOR_PPI, "o", "-"),
        ("ppi_weighted_coverage", "PPI-W", COLOR_PPI_W, "o", "-"),
        ("proxy_adj_coverage", "Proxy+Adj", COLOR_PROXY, "s", "--"),
        ("ppi_adj_coverage", "PPI+Adj", COLOR_PPI, "s", "--"),
        ("ppi_weighted_adj_coverage", "PPI-W+Adj", COLOR_PPI_W, "s", "--"),
        ("proxy_domain_boot_coverage", "Proxy+Boot", COLOR_PROXY, "D", ":"),
        ("ppi_domain_boot_coverage", "PPI+Boot", COLOR_PPI, "D", ":"),
        ("ppi_weighted_domain_boot_coverage", "PPI-W+Boot", COLOR_PPI_W, "D", ":"),
    ]

    kappa_colors = plt.cm.viridis(np.linspace(0, 0.9, len(KAPPA_VALUES)))

    n_methods = len(methods)
    n_cols = 3
    n_rows = (n_methods + n_cols - 1) // n_cols

    # Create one figure per n_domains value
    for n_domains_idx, n_domains in enumerate(N_DOMAINS_VALUES):
        fig, axes = plt.subplots(
            n_rows, n_cols, figsize=(12, 8), squeeze=False, facecolor="w"
        )
        axes = axes.flatten()

        for idx, (col, title, base_color, marker, linestyle) in enumerate(methods):
            ax = axes[idx]

            for i, kappa in enumerate(KAPPA_VALUES):
                df_subset = df[
                    (df["kappa"] == kappa) & (df["n_domains"] == n_domains)
                ].sort_values("sample_size")
                ax.plot(
                    df_subset["sample_size"],
                    df_subset[col],
                    label=f"$\\it{{\\kappa}}={kappa}$",
                    color=kappa_colors[i],
                    marker=marker,
                    markersize=5,
                    linewidth=1.5,
                    linestyle=linestyle,
                )

            ax.axhline(
                y=0.95, color="gray", linestyle="--", linewidth=1, label="Nominal (95%)"
            )
            ax.set_xscale("log")
            ax.set_ylim(0.0, 1.05)
            ax.grid(True, alpha=0.3)
            ax.set_title(title, fontsize=13, color=base_color, fontweight="bold")

        for idx in range(len(methods), len(axes)):
            axes[idx].set_visible(False)

        fig.text(
            0.5,
            0.01,
            "Sample Size per Domain",
            ha="center",
            fontsize=14,
            fontweight="bold",
        )
        fig.text(
            0.01,
            0.5,
            "Coverage Rate",
            va="center",
            rotation="vertical",
            fontsize=14,
            fontweight="bold",
        )

        handles, labels = ax.get_legend_handles_labels()
        fig.legend(
            handles,
            labels,
            loc="upper center",
            ncol=4,
            fontsize=9,
            bbox_to_anchor=(0.5, 0.98),
        )

        fig.tight_layout(rect=[0.02, 0.02, 1, 0.95])

        fig_path = save_dir / f"coverage_by_method_K{n_domains}.png"
        plt.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="w")
        print(f"Saved method comparison plot: {fig_path}")

        plt.show()


def plot_base_vs_adjusted(df: pd.DataFrame, save_dir: Path = OUTPUT_DIR):
    """
    Create comparison plots showing base methods vs their MoM and Domain Bootstrap adjusted versions.
    One plot per n_domains value.
    """
    for n_domains in N_DOMAINS_VALUES:
        comparisons = [
            (
                "ppi_coverage",
                "ppi_adj_coverage",
                "ppi_domain_boot_coverage",
                "PPI",
                COLOR_PPI,
            ),
            (
                "ppi_weighted_coverage",
                "ppi_weighted_adj_coverage",
                "ppi_weighted_domain_boot_coverage",
                "PPI-W",
                COLOR_PPI_W,
            ),
        ]

        fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor="w")

        for idx, (base_col, adj_col, boot_col, title, color) in enumerate(comparisons):
            ax = axes[idx]

            for kappa in KAPPA_VALUES:
                df_subset = df[
                    (df["kappa"] == kappa) & (df["n_domains"] == n_domains)
                ].sort_values("sample_size")

                ax.plot(
                    df_subset["sample_size"],
                    df_subset[base_col],
                    color=color,
                    linestyle="-",
                    marker="o",
                    markersize=4,
                    linewidth=1.5,
                    alpha=0.4,
                    label=f"Base ($\\it{{\\kappa}}={kappa}$)" if kappa == KAPPA_VALUES[0] else None,
                )
                ax.plot(
                    df_subset["sample_size"],
                    df_subset[adj_col],
                    color=color,
                    linestyle="--",
                    marker="s",
                    markersize=4,
                    linewidth=2,
                    label=f"Adj ($\\it{{\\kappa}}={kappa}$)" if kappa == KAPPA_VALUES[0] else None,
                )
                ax.plot(
                    df_subset["sample_size"],
                    df_subset[boot_col],
                    color=color,
                    linestyle=":",
                    marker="D",
                    markersize=4,
                    linewidth=1.5,
                    alpha=0.7,
                    label=f"Boot ($\\it{{\\kappa}}={kappa}$)" if kappa == KAPPA_VALUES[0] else None,
                )

            ax.axhline(y=0.95, color="gray", linestyle=":", linewidth=1.5)
            ax.set_xscale("log")
            ax.set_ylim(0.0, 1.05)
            ax.grid(True, alpha=0.3)
            ax.set_title(
                f"{title}: Base vs Adj vs Boot",
                fontsize=13,
                color=color,
                fontweight="bold",
            )

        fig.text(
            0.5,
            -0.05,
            "Sample Size per Domain",
            ha="center",
            fontsize=14,
            fontweight="bold",
        )
        fig.text(
            0.01,
            0.5,
            "Coverage Rate",
            va="center",
            rotation="vertical",
            fontsize=14,
            fontweight="bold",
        )

        handles, labels = ax.get_legend_handles_labels()
        fig.legend(
            handles,
            labels,
            loc="upper center",
            ncol=4,
            fontsize=9,
            bbox_to_anchor=(0.5, 1.0),
        )

        fig.tight_layout(rect=[0.02, 0.02, 1, 0.95])

        fig_path = save_dir / f"coverage_base_vs_adjusted_K{n_domains}.png"
        plt.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="w")
        print(f"Saved base vs adjusted comparison plot: {fig_path}")

        plt.show()


def plot_adj_vs_domboot_by_n_domains(df: pd.DataFrame, save_dir: Path = OUTPUT_DIR):
    """
    Compare MoM adjustment vs Domain Bootstrap methods across different n_domains.
    Shows how the gap between methods changes with the number of source domains.
    """
    methods = [
        ("ppi_adj_coverage", "ppi_domain_boot_coverage", "PPI", COLOR_PPI),
        (
            "ppi_weighted_adj_coverage",
            "ppi_weighted_domain_boot_coverage",
            "PPI-W",
            COLOR_PPI_W,
        ),
    ]

    n_rows = len(methods)
    n_cols = len(KAPPA_VALUES)

    fig, axes = plt.subplots(
        n_rows, n_cols, figsize=(12, 8), squeeze=False, facecolor="w"
    )

    n_domain_colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(N_DOMAINS_VALUES)))

    for i, (adj_col, boot_col, method_name, base_color) in enumerate(methods):
        for j, kappa in enumerate(KAPPA_VALUES):
            ax = axes[i, j]

            for k, n_domains in enumerate(N_DOMAINS_VALUES):
                df_subset = df[
                    (df["kappa"] == kappa) & (df["n_domains"] == n_domains)
                ].sort_values("sample_size")

                ax.plot(
                    df_subset["sample_size"],
                    df_subset[adj_col],
                    color=n_domain_colors[k],
                    linestyle="--",
                    marker="s",
                    markersize=5,
                    linewidth=2,
                    label=f"Adj K={n_domains}" if j == 0 else None,
                )
                ax.plot(
                    df_subset["sample_size"],
                    df_subset[boot_col],
                    color=n_domain_colors[k],
                    linestyle=":",
                    marker="D",
                    markersize=5,
                    linewidth=2,
                    alpha=0.7,
                    label=f"Boot K={n_domains}" if j == 0 else None,
                )

            ax.axhline(y=0.95, color="gray", linestyle=":", linewidth=1.5)
            ax.set_xscale("log")
            ax.set_ylim(0.0, 1.05)
            ax.grid(True, alpha=0.3)
            ax.set_title(
                f"{method_name}: $\\it{{\\kappa}}={kappa}$",
                fontsize=13,
                color=base_color,
                fontweight="bold",
            )

    fig.text(0.5, 0.01, "Sample Size", ha="center", fontsize=14, fontweight="bold")
    fig.text(
        0.01,
        0.5,
        "Coverage Rate",
        va="center",
        rotation="vertical",
        fontsize=14,
        fontweight="bold",
    )

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=4,
        fontsize=9,
        bbox_to_anchor=(0.5, 1.02),
    )

    fig.tight_layout(rect=[0.02, 0.02, 1, 0.95])

    fig_path = save_dir / "adj_vs_domboot_by_n_domains.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight", facecolor="w")
    print(f"Saved adjustment comparison plot: {fig_path}")

    plt.show()


# Generate method-centric plots
plot_coverage_by_method(df_coverage_summary)

# Generate base vs adjusted comparison
plot_base_vs_adjusted(df_coverage_summary)

# Generate adjustment vs domain bootstrap comparison
plot_adj_vs_domboot_by_n_domains(df_coverage_summary)

In [ ]:
# =============================================================================
# CI LENGTH PLOTS - Compare CI widths across methods
# =============================================================================
# Standardized color scheme:
#   - Primary: Blue (#1f77b4)
#   - Proxy: Red (#d62728)
#   - PPI: Orange (#ff7f0e)
#   - PPI-W: Purple (#9467bd)
# Standardized line/marker scheme:
#   - Base methods: solid line (-), circle marker (o)
#   - Adjusted (+Adj): dashed line (--), square marker (s)
#   - Bootstrap (+Boot): dotted line (:), diamond marker (D)

# Color constants
COLOR_PRIMARY = "#1f77b4"  # Blue
COLOR_PROXY = "#d62728"    # Red
COLOR_PPI = "#ff7f0e"      # Orange
COLOR_PPI_W = "#9467bd"    # Purple

def plot_ci_length_by_sample_size(df: pd.DataFrame, save_dir: Path = OUTPUT_DIR):
    """
    Create CI length vs sample size plots for each (n_domains, kappa) combination.

    Shows how CI width varies with sample size for all methods.
    """
    methods = [
        # Base methods - solid line, circle marker
        ("avg_primary_ci_length", "Primary Only", COLOR_PRIMARY, "o", "-"),
        ("avg_proxy_ci_length", "Proxy Only", COLOR_PROXY, "o", "-"),
        ("avg_ppi_ci_length", "PPI", COLOR_PPI, "o", "-"),
        ("avg_ppi_weighted_ci_length", "PPI-W", COLOR_PPI_W, "o", "-"),
        # MoM-adjusted methods - dashed line, square marker
        ("avg_proxy_adj_ci_length", "Proxy+Adj", COLOR_PROXY, "s", "--"),
        ("avg_ppi_adj_ci_length", "PPI+Adj", COLOR_PPI, "s", "--"),
        ("avg_ppi_weighted_adj_ci_length", "PPI-W+Adj", COLOR_PPI_W, "s", "--"),
        # Domain bootstrap methods - dotted line, diamond marker
        ("avg_proxy_domain_boot_ci_length", "Proxy+Boot", COLOR_PROXY, "D", ":"),
        ("avg_ppi_domain_boot_ci_length", "PPI+Boot", COLOR_PPI, "D", ":"),
        ("avg_ppi_weighted_domain_boot_ci_length", "PPI-W+Boot", COLOR_PPI_W, "D", ":"),
    ]

    # Create individual plots for each (n_domains, kappa) combination (only for a few key combinations)
    for n_domains in [N_DOMAINS_VALUES[0], N_DOMAINS_VALUES[-1]]:  # Just first and last
        for kappa in [KAPPA_VALUES[0], KAPPA_VALUES[1]]:  # Just first and second
            df_subset = df[(df["n_domains"] == n_domains) & (df["kappa"] == kappa)].sort_values("sample_size")

            fig, ax = plt.subplots(figsize=(14, 8))

            for col, label, color, marker, linestyle in methods:
                ax.plot(
                    df_subset["sample_size"],
                    df_subset[col],
                    label=label,
                    color=color,
                    marker=marker,
                    markersize=8,
                    linewidth=2,
                    linestyle=linestyle,
                )

            ax.set_xlabel("Sample Size per Domain", fontsize=12)
            ax.set_ylabel("Average CI Length", fontsize=12)
            ax.set_title(f"95% CI Length vs Sample Size (K={n_domains}, κ={kappa})", fontsize=14)
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.legend(loc="upper right", fontsize=8, ncol=2)
            ax.grid(True, alpha=0.3)

            plt.tight_layout()

            # Save figure
            fig_path = save_dir / f"ci_length_vs_sample_size_K{n_domains}_kappa_{kappa}.png"
            plt.savefig(fig_path, dpi=150, bbox_inches="tight")
            print(f"Saved CI length plot: {fig_path}")

            plt.show()


def plot_ci_length_combined(df: pd.DataFrame, save_dir: Path = OUTPUT_DIR):
    """
    Create a combined figure with CI lengths by n_domains and kappa.
    """
    methods = [
        # PPI base - solid line, circle marker
        ("avg_ppi_ci_length", "PPI", COLOR_PPI, "o", "-"),
        # PPI-W base - solid line, circle marker
        ("avg_ppi_weighted_ci_length", "PPI-W", COLOR_PPI_W, "o", "-"),
        # PPI adjusted - dashed line, square marker
        ("avg_ppi_adj_ci_length", "PPI+Adj", COLOR_PPI, "s", "--"),
        # PPI-W adjusted - dashed line, square marker
        ("avg_ppi_weighted_adj_ci_length", "PPI-W+Adj", COLOR_PPI_W, "s", "--"),
        # PPI bootstrap - dotted line, diamond marker
        ("avg_ppi_domain_boot_ci_length", "PPI+Boot", COLOR_PPI, "D", ":"),
        # PPI-W bootstrap - dotted line, diamond marker
        ("avg_ppi_weighted_domain_boot_ci_length", "PPI-W+Boot", COLOR_PPI_W, "D", ":"),
    ]

    n_rows = len(N_DOMAINS_VALUES)
    n_cols = len(KAPPA_VALUES)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 3.5*n_rows), squeeze=False)

    for i, n_domains in enumerate(N_DOMAINS_VALUES):
        for j, kappa in enumerate(KAPPA_VALUES):
            ax = axes[i, j]
            df_subset = df[(df["n_domains"] == n_domains) & (df["kappa"] == kappa)].sort_values("sample_size")

            for col, label, color, marker, linestyle in methods:
                ax.plot(
                    df_subset["sample_size"],
                    df_subset[col],
                    label=label,
                    color=color,
                    marker=marker,
                    markersize=4,
                    linewidth=1.5,
                    linestyle=linestyle,
                )

            ax.set_xlabel("Sample Size", fontsize=8)
            ax.set_ylabel("Avg CI Length", fontsize=8)
            ax.set_title(f"K={n_domains}, κ={kappa}", fontsize=9)
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.grid(True, alpha=0.3)

    # Add legend to the figure
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=8, bbox_to_anchor=(0.5, -0.04))

    fig.suptitle("95% CI Length: Rows=Domains, Columns=Kappa", fontsize=14, y=1.02)
    plt.tight_layout()

    # Save combined figure
    fig_path = save_dir / "ci_length_grid_n_domains_kappa.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"Saved combined CI length plot: {fig_path}")

    plt.show()


def plot_coverage_vs_ci_length(df: pd.DataFrame, save_dir: Path = OUTPUT_DIR):
    """
    Create scatter plots of coverage vs CI length for each method.

    This helps visualize the trade-off between coverage and precision.
    """
    methods = [
        ("ppi", "ppi_coverage", "avg_ppi_ci_length", "PPI", COLOR_PPI),
        ("ppi_adj", "ppi_adj_coverage", "avg_ppi_adj_ci_length", "PPI+Adj", COLOR_PPI),
        ("ppi_domain_boot", "ppi_domain_boot_coverage", "avg_ppi_domain_boot_ci_length", "PPI+Boot", COLOR_PPI),
        ("ppi_weighted_adj", "ppi_weighted_adj_coverage", "avg_ppi_weighted_adj_ci_length", "PPI-W+Adj", COLOR_PPI_W),
    ]

    n_domain_markers = ["o", "s", "^", "D"]
    n_domain_colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(N_DOMAINS_VALUES)))

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()

    for idx, (method_id, cov_col, len_col, title, color) in enumerate(methods):
        ax = axes[idx]

        for i, n_domains in enumerate(N_DOMAINS_VALUES):
            for j, kappa in enumerate(KAPPA_VALUES):
                df_subset = df[(df["n_domains"] == n_domains) & (df["kappa"] == kappa)]
                ax.scatter(
                    df_subset[len_col],
                    df_subset[cov_col],
                    c=[n_domain_colors[i]],
                    marker=n_domain_markers[i],
                    label=f"K={n_domains}" if j == 0 else None,
                    alpha=0.7,
                    s=60,
                    edgecolors="white",
                )

        ax.axhline(y=0.95, color="red", linestyle="--", linewidth=1.5, alpha=0.7, label="Nominal (95%)")
        ax.set_xlabel("Average CI Length", fontsize=11)
        ax.set_ylabel("Coverage Rate", fontsize=11)
        ax.set_title(title, fontsize=13, color=color)
        ax.set_ylim(0.7, 1.0)
        ax.legend(loc="lower right", fontsize=9)
        ax.grid(True, alpha=0.3)

    fig.suptitle("Coverage vs CI Length Trade-off by Number of Domains", fontsize=14)
    plt.tight_layout()

    fig_path = save_dir / "coverage_vs_ci_length.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"Saved coverage vs CI length plot: {fig_path}")

    plt.show()


# Generate CI length plots
plot_ci_length_by_sample_size(df_coverage_summary)
plot_ci_length_combined(df_coverage_summary)
plot_coverage_vs_ci_length(df_coverage_summary)


In [ ]:
# =============================================================================
# Publication version CI LENGTH PLOTS - Compare CI widths across methods
# =============================================================================
# Standardized color scheme:
#   - Primary: Blue (#1f77b4)
#   - Proxy: Red (#d62728)
#   - PPI: Orange (#ff7f0e)
#   - PPI-W: Purple (#9467bd)
# Standardized line/marker scheme:
#   - Base methods: solid line (-), circle marker (o)
#   - Adjusted (+Adj): dashed line (--), square marker (s)
#   - Bootstrap (+Boot): dotted line (:), diamond marker (D)

# Color constants
COLOR_PRIMARY = "#1f77b4"  # Blue
COLOR_PROXY = "#d62728"    # Red
COLOR_PPI = "#ff7f0e"      # Orange
COLOR_PPI_W = "#9467bd"    # Purple

def plot_ci_length_combined(df: pd.DataFrame, save_dir: Path = OUTPUT_DIR):
    """
    Create a combined figure with CI lengths by n_domains and kappa.
    Show results for only the largest and smallest values of n_domains and kappa.
    """
    methods = [
        # PPI base - solid line, circle marker
        ("avg_ppi_ci_length", "PPI", COLOR_PPI, "o", "-"),
        # PPI adjusted - dashed line, square marker
        ("avg_ppi_adj_ci_length", "PPI+Adj", COLOR_PPI, "s", "--"),
        # PPI bootstrap - dotted line, diamond marker
        ("avg_ppi_domain_boot_ci_length", "PPI+Boot", COLOR_PPI, "D", ":"),
        # PPI-W base - solid line, circle marker
        ("avg_ppi_weighted_ci_length", "PPI-W", COLOR_PPI_W, "o", "-"),
        # PPI-W adjusted - dashed line, square marker
        ("avg_ppi_weighted_adj_ci_length", "PPI-W+Adj", COLOR_PPI_W, "s", "--"),
        # PPI-W bootstrap - dotted line, diamond marker
        ("avg_ppi_weighted_domain_boot_ci_length", "PPI-W+Boot", COLOR_PPI_W, "D", ":"),
        # Proxy base - solid line, circle marker
        ("avg_proxy_ci_length", "Proxy Only", COLOR_PROXY, "o", "-"),
        # Proxy adjusted - dashed line, square marker
        ("avg_proxy_adj_ci_length", "Proxy+Adj", COLOR_PROXY, "s", "--"),
        # Proxy bootstrap - dotted line, diamond marker
        ("avg_proxy_domain_boot_ci_length", "Proxy+Boot", COLOR_PROXY, "D", ":"),
    ]

    # Use only the smallest and largest values for n_domains and kappa
    selected_n_domains = [N_DOMAINS_VALUES[0], N_DOMAINS_VALUES[-1]]
    selected_kappa = [KAPPA_VALUES[0], KAPPA_VALUES[-1]]

    n_rows = len(selected_n_domains)
    n_cols = len(selected_kappa)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 3.5*n_rows), squeeze=False, facecolor='w')

    for i, n_domains in enumerate(selected_n_domains):
        for j, kappa in enumerate(selected_kappa):
            ax = axes[i, j]
            df_subset = df[(df["n_domains"] == n_domains) & (df["kappa"] == kappa)].sort_values("sample_size")

            for col, label, color, marker, linestyle in methods:
                ax.plot(
                    df_subset["sample_size"],
                    df_subset[col],
                    label=label,
                    color=color,
                    marker=marker,
                    markersize=4,
                    linewidth=1.5,
                    linestyle=linestyle,
                )

            ax.set_xlabel("Sample Size", fontsize=10, fontweight="bold")
            ax.set_ylabel("Average CI Length", fontsize=10, fontweight="bold")
            ax.set_title(f"N_domains={n_domains}, κ={kappa}", fontsize=12, fontweight="bold")
            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.grid(True, alpha=0.3)

    # Add legend to the figure above the plots
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3, fontsize=9, bbox_to_anchor=(0.5, 1.08))

    fig.suptitle("95% CI Length: Rows=Domains, Columns=Kappa", fontsize=16, y=1.12, fontweight="bold")
    plt.tight_layout()

    # Save combined figure
    fig_path = save_dir / "ci_length_grid_n_domains_kappa.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    print(f"Saved combined CI length plot: {fig_path}")

    plt.show()


# Generate CI length plots

plot_ci_length_combined(df_coverage_summary)

In [ ]:
# =============================================================================
# TIMING VISUALIZATION PLOTS
# =============================================================================

def plot_timing_breakdown(df: pd.DataFrame, save_dir: Path = OUTPUT_DIR):
    """
    Create timing breakdown plots to identify bottlenecks.
    """
    timing_methods = [
        ("avg_time_data_generation", "Data Generation", "#1f77b4"),
        ("avg_time_primary_estimate", "Primary Est", "#ff7f0e"),
        ("avg_time_proxy_estimate", "Proxy Est", "#2ca02c"),
        ("avg_time_ppi_estimate", "PPI Est", "#d62728"),
        ("avg_time_ppi_weighted_estimate", "PPI-W Est", "#9467bd"),
        ("avg_time_proxy_adj_estimate", "Proxy+Adj", "#8c564b"),
        ("avg_time_ppi_adj_estimate", "PPI+Adj", "#e377c2"),
        ("avg_time_ppi_weighted_adj_estimate", "PPI-W+Adj", "#7f7f7f"),
        ("avg_time_proxy_domain_boot_estimate", "Proxy+Boot", "#bcbd22"),
        ("avg_time_ppi_domain_boot_estimate", "PPI+Boot", "#17becf"),
        ("avg_time_ppi_weighted_domain_boot_estimate", "PPI-W+Boot", "#aec7e8"),
    ]

    # 1. Stacked bar chart showing timing breakdown by sample size
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: By sample size (averaged across n_domains and kappa)
    ax = axes[0]
    timing_by_n = []
    for sample_size in SAMPLE_SIZES:
        df_n = df[df["sample_size"] == sample_size]
        row = {"n": sample_size}
        for col, name, color in timing_methods:
            row[name] = df_n[col].mean() * 1000  # Convert to ms
        timing_by_n.append(row)

    df_timing = pd.DataFrame(timing_by_n)

    # Create stacked bar
    bottom = np.zeros(len(SAMPLE_SIZES))
    x = np.arange(len(SAMPLE_SIZES))

    for col, name, color in timing_methods:
        values = df_timing[name].values
        ax.bar(x, values, bottom=bottom, label=name, color=color, width=0.8)
        bottom += values

    ax.set_xlabel("Sample Size", fontsize=12)
    ax.set_ylabel("Time (ms)", fontsize=12)
    ax.set_title("Timing Breakdown by Sample Size", fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels([str(n) for n in SAMPLE_SIZES], rotation=45, ha="right")
    ax.legend(loc="upper left", fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3, axis="y")

    # Right: By number of domains (averaged across sample_size and kappa)
    ax = axes[1]
    timing_by_k = []
    for n_domains in N_DOMAINS_VALUES:
        df_k = df[df["n_domains"] == n_domains]
        row = {"K": n_domains}
        for col, name, color in timing_methods:
            row[name] = df_k[col].mean() * 1000  # Convert to ms
        timing_by_k.append(row)

    df_timing_k = pd.DataFrame(timing_by_k)

    # Create stacked bar
    bottom = np.zeros(len(N_DOMAINS_VALUES))
    x = np.arange(len(N_DOMAINS_VALUES))

    for col, name, color in timing_methods:
        values = df_timing_k[name].values
        ax.bar(x, values, bottom=bottom, label=name, color=color, width=0.6)
        bottom += values

    ax.set_xlabel("Number of Domains (K)", fontsize=12)
    ax.set_ylabel("Time (ms)", fontsize=12)
    ax.set_title("Timing Breakdown by Number of Domains", fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels([str(k) for k in N_DOMAINS_VALUES])
    ax.legend(loc="upper left", fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()

    fig_path = save_dir / "timing_breakdown_stacked.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"Saved timing breakdown plot: {fig_path}")

    plt.show()

    # 2. Line plot showing timing vs sample size for each method
    fig, ax = plt.subplots(figsize=(14, 8))

    for col, name, color in timing_methods:
        timing_by_n = []
        for sample_size in SAMPLE_SIZES:
            df_n = df[df["sample_size"] == sample_size]
            timing_by_n.append(df_n[col].mean() * 1000)

        ax.plot(SAMPLE_SIZES, timing_by_n, label=name, color=color, marker="o", markersize=6, linewidth=2)

    ax.set_xlabel("Sample Size per Domain", fontsize=12)
    ax.set_ylabel("Average Time (ms)", fontsize=12)
    ax.set_title("Method Timing vs Sample Size", fontsize=14)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.legend(loc="upper left", fontsize=9, ncol=2)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()

    fig_path = save_dir / "timing_vs_sample_size.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"Saved timing vs sample size plot: {fig_path}")

    plt.show()

    # 3. Pie chart showing overall timing breakdown
    fig, ax = plt.subplots(figsize=(10, 10))

    overall_timing = {}
    for col, name, color in timing_methods:
        overall_timing[name] = df[col].mean() * 1000

    # Sort by time
    sorted_items = sorted(overall_timing.items(), key=lambda x: x[1], reverse=True)
    labels = [item[0] for item in sorted_items]
    sizes = [item[1] for item in sorted_items]
    colors = [next(c for col, n, c in timing_methods if n == label) for label in labels]

    # Only show labels for slices > 2%
    total = sum(sizes)
    def autopct_func(pct):
        return f'{pct:.1f}%' if pct > 2 else ''

    wedges, texts, autotexts = ax.pie(
        sizes,
        labels=labels,
        colors=colors,
        autopct=autopct_func,
        startangle=90,
        pctdistance=0.75
    )

    ax.set_title("Overall Timing Breakdown", fontsize=14)

    plt.tight_layout()

    fig_path = save_dir / "timing_pie_chart.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"Saved timing pie chart: {fig_path}")

    plt.show()

    # 4. Heatmap of timing by (n_domains, sample_size) for domain bootstrap
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    boot_methods = [
        ("avg_time_proxy_domain_boot_estimate", "Proxy + Domain Bootstrap"),
        ("avg_time_ppi_domain_boot_estimate", "PPI + Domain Bootstrap"),
        ("avg_time_ppi_weighted_domain_boot_estimate", "PPI-W + Domain Bootstrap"),
    ]

    for idx, (col, title) in enumerate(boot_methods):
        ax = axes[idx]

        # Create pivot table
        pivot = df.pivot_table(
            values=col,
            index="sample_size",
            columns="n_domains",
            aggfunc="mean"
        ) * 1000  # Convert to ms

        im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd")

        ax.set_xticks(range(len(N_DOMAINS_VALUES)))
        ax.set_xticklabels([str(k) for k in N_DOMAINS_VALUES])
        ax.set_yticks(range(len(SAMPLE_SIZES)))
        ax.set_yticklabels([str(n) for n in SAMPLE_SIZES])
        ax.set_xlabel("Number of Domains (K)")
        ax.set_ylabel("Sample Size")
        ax.set_title(f"{title}\n(ms)")

        # Add colorbar
        plt.colorbar(im, ax=ax, shrink=0.8)

        # Add text annotations
        for i in range(len(SAMPLE_SIZES)):
            for j in range(len(N_DOMAINS_VALUES)):
                text = ax.text(j, i, f"{pivot.values[i, j]:.1f}",
                              ha="center", va="center", color="black", fontsize=8)

    plt.tight_layout()

    fig_path = save_dir / "timing_heatmap_bootstrap.png"
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"Saved timing heatmap: {fig_path}")

    plt.show()


# Generate timing plots
plot_timing_breakdown(df_coverage_summary)

In [ ]:
# =============================================================================
# SUMMARY STATISTICS
# =============================================================================

print("\n" + "="*80)
print("SIMULATION SUMMARY")
print("="*80)

print(f"\nTotal simulations run: {len(all_results):,}")
print(f"Number of domains (K): {N_DOMAINS_VALUES}")
print(f"Kappa values: {KAPPA_VALUES}")
print(f"Sample sizes: {list(SAMPLE_SIZES)}")
print(f"Simulations per combination: {N_SIMULATIONS}")
print(f"Total combinations: {len(N_DOMAINS_VALUES)} x {len(KAPPA_VALUES)} x {len(SAMPLE_SIZES)} = {len(N_DOMAINS_VALUES) * len(KAPPA_VALUES) * len(SAMPLE_SIZES)}")

# =============================================================================
# TIMING BREAKDOWN
# =============================================================================

print("\n" + "="*80)
print("TIMING BREAKDOWN (BOTTLENECK ANALYSIS)")
print("="*80)

# Overall timing summary
timing_cols = [
    ("avg_time_data_generation", "Data Generation"),
    ("avg_time_primary_estimate", "Primary Estimate"),
    ("avg_time_proxy_estimate", "Proxy Estimate"),
    ("avg_time_ppi_estimate", "PPI Estimate"),
    ("avg_time_ppi_weighted_estimate", "PPI Weighted Est"),
    ("avg_time_proxy_adj_estimate", "Proxy + Adj"),
    ("avg_time_ppi_adj_estimate", "PPI + Adj"),
    ("avg_time_ppi_weighted_adj_estimate", "PPI-W + Adj"),
    ("avg_time_proxy_domain_boot_estimate", "Proxy + DomBoot"),
    ("avg_time_ppi_domain_boot_estimate", "PPI + DomBoot"),
    ("avg_time_ppi_weighted_domain_boot_estimate", "PPI-W + DomBoot"),
    ("avg_time_total", "TOTAL"),
]

print("\n" + "-"*80)
print("Average Time per Simulation (milliseconds) - Overall")
print("-"*80)

overall_timing = {}
for col, name in timing_cols:
    avg_time_ms = df_coverage_summary[col].mean() * 1000
    overall_timing[name] = avg_time_ms
    pct = (avg_time_ms / (df_coverage_summary["avg_time_total"].mean() * 1000)) * 100 if name != "TOTAL" else 100.0
    print(f"  {name:26s}: {avg_time_ms:8.2f} ms  ({pct:5.1f}%)")

# Timing by n_domains
print("\n" + "-"*80)
print("Average Time per Simulation by Number of Domains (milliseconds)")
print("-"*80)

timing_by_k = []
for n_domains in N_DOMAINS_VALUES:
    df_k = df_coverage_summary[df_coverage_summary["n_domains"] == n_domains]
    row = {"K": n_domains}
    for col, name in timing_cols:
        row[name] = df_k[col].mean() * 1000
    timing_by_k.append(row)

df_timing_k = pd.DataFrame(timing_by_k)
print("\n" + df_timing_k.to_string(index=False))

# Timing by sample_size
print("\n" + "-"*80)
print("Average Time per Simulation by Sample Size (milliseconds)")
print("-"*80)

timing_by_n = []
for sample_size in SAMPLE_SIZES:
    df_n = df_coverage_summary[df_coverage_summary["sample_size"] == sample_size]
    row = {"n": sample_size}
    row["DataGen"] = df_n["avg_time_data_generation"].mean() * 1000
    row["Base Methods"] = (df_n["avg_time_primary_estimate"].mean() +
                           df_n["avg_time_proxy_estimate"].mean() +
                           df_n["avg_time_ppi_estimate"].mean() +
                           df_n["avg_time_ppi_weighted_estimate"].mean()) * 1000
    row["MoM Adj"] = (df_n["avg_time_proxy_adj_estimate"].mean() +
                      df_n["avg_time_ppi_adj_estimate"].mean() +
                      df_n["avg_time_ppi_weighted_adj_estimate"].mean()) * 1000
    row["DomBoot"] = (df_n["avg_time_proxy_domain_boot_estimate"].mean() +
                      df_n["avg_time_ppi_domain_boot_estimate"].mean() +
                      df_n["avg_time_ppi_weighted_domain_boot_estimate"].mean()) * 1000
    row["TOTAL"] = df_n["avg_time_total"].mean() * 1000
    timing_by_n.append(row)

df_timing_n = pd.DataFrame(timing_by_n)
print("\n" + df_timing_n.to_string(index=False))

# Identify bottlenecks
print("\n" + "-"*80)
print("BOTTLENECK IDENTIFICATION")
print("-"*80)

# Sort by time
sorted_timing = sorted([(name, t) for name, t in overall_timing.items() if name != "TOTAL"],
                       key=lambda x: x[1], reverse=True)

print("\nMethods sorted by average time (slowest first):")
total_time = overall_timing["TOTAL"]
cumulative = 0
for name, t in sorted_timing:
    cumulative += t
    pct = (t / total_time) * 100
    cum_pct = (cumulative / total_time) * 100
    print(f"  {name:26s}: {t:8.2f} ms  ({pct:5.1f}%)  [cumulative: {cum_pct:5.1f}%]")

# Summary by n_domains
for n_domains in N_DOMAINS_VALUES:
    print("\n" + "="*80)
    print(f"RESULTS FOR K={n_domains} DOMAINS")
    print("="*80)

    df_k = df_coverage_summary[df_coverage_summary["n_domains"] == n_domains]

    print("\n" + "-"*80)
    print("Average Coverage by Method (across all kappa and sample sizes):")
    print("-"*80)

    # Base methods
    print("\nBase Methods:")
    base_methods = [
        ("Primary Only", "primary_coverage"),
        ("Proxy Only", "proxy_coverage"),
        ("PPI", "ppi_coverage"),
        ("PPI + Weighting", "ppi_weighted_coverage"),
    ]
    for name, col in base_methods:
        coverage = df_k[col].mean()
        status = "✓" if abs(coverage - 0.95) < 0.05 else "✗"
        print(f"  {name:26s}: {coverage:.3f} {status}")

    # MoM-adjusted methods
    print("\nMoM-Adjusted Methods:")
    mom_methods = [
        ("Proxy + Adj", "proxy_adj_coverage"),
        ("PPI + Adj", "ppi_adj_coverage"),
        ("PPI + W + Adj", "ppi_weighted_adj_coverage"),
    ]
    for name, col in mom_methods:
        coverage = df_k[col].mean()
        status = "✓" if abs(coverage - 0.95) < 0.03 else "✗"
        print(f"  {name:26s}: {coverage:.3f} {status}")

    # Domain bootstrap methods
    print("\nDomain Bootstrap Methods (estimate-level resampling):")
    boot_methods = [
        ("Proxy + Dom Bootstrap", "proxy_domain_boot_coverage"),
        ("PPI + Dom Bootstrap", "ppi_domain_boot_coverage"),
        ("PPI + W + Dom Bootstrap", "ppi_weighted_domain_boot_coverage"),
    ]
    for name, col in boot_methods:
        coverage = df_k[col].mean()
        status = "✓" if abs(coverage - 0.95) < 0.03 else "✗"
        print(f"  {name:26s}: {coverage:.3f} {status}")

# Overall comparison across n_domains
print("\n" + "="*80)
print("COMPARISON ACROSS NUMBER OF DOMAINS")
print("="*80)

print("\n" + "-"*80)
print("Average Coverage by Method and Number of Domains:")
print("-"*80)

comparison_data = []
for n_domains in N_DOMAINS_VALUES:
    df_k = df_coverage_summary[df_coverage_summary["n_domains"] == n_domains]
    comparison_data.append({
        "K": n_domains,
        "PPI": df_k["ppi_coverage"].mean(),
        "PPI+Adj": df_k["ppi_adj_coverage"].mean(),
        "PPI+DomBoot": df_k["ppi_domain_boot_coverage"].mean(),
        "PPI-W": df_k["ppi_weighted_coverage"].mean(),
        "PPI-W+Adj": df_k["ppi_weighted_adj_coverage"].mean(),
        "PPI-W+DomBoot": df_k["ppi_weighted_domain_boot_coverage"].mean(),
    })
df_comparison = pd.DataFrame(comparison_data)
print("\n" + df_comparison.to_string(index=False))

print("\n" + "-"*80)
print("Average CI Length by Method and Number of Domains:")
print("-"*80)

ci_comparison = []
for n_domains in N_DOMAINS_VALUES:
    df_k = df_coverage_summary[df_coverage_summary["n_domains"] == n_domains]
    ci_comparison.append({
        "K": n_domains,
        "PPI": df_k["avg_ppi_ci_length"].mean(),
        "PPI+Adj": df_k["avg_ppi_adj_ci_length"].mean(),
        "PPI+DomBoot": df_k["avg_ppi_domain_boot_ci_length"].mean(),
        "PPI-W+Adj": df_k["avg_ppi_weighted_adj_ci_length"].mean(),
        "PPI-W+DomBoot": df_k["avg_ppi_weighted_domain_boot_ci_length"].mean(),
    })
df_ci_comparison = pd.DataFrame(ci_comparison)
print("\n" + df_ci_comparison.to_string(index=False))

print("\n" + "-"*80)
print("Coverage at Largest Sample Size (n=10000) by Kappa and K:")
print("-"*80)

df_max_n = df_coverage_summary[df_coverage_summary["sample_size"] == SAMPLE_SIZES[-1]]

print("\nPPI + Adjustment Coverage:")
pivot_ppi_adj = df_max_n.pivot_table(
    values="ppi_adj_coverage",
    index="kappa",
    columns="n_domains"
)
print(pivot_ppi_adj.to_string())

print("\nPPI + Domain Bootstrap Coverage:")
pivot_ppi_boot = df_max_n.pivot_table(
    values="ppi_domain_boot_coverage",
    index="kappa",
    columns="n_domains"
)
print(pivot_ppi_boot.to_string())

print("\n" + "-"*80)
print("Average Gamma^2 / Rho Estimates by K:")
print("-"*80)

for n_domains in N_DOMAINS_VALUES:
    df_k = df_coverage_summary[df_coverage_summary["n_domains"] == n_domains]
    print(f"\nK={n_domains} domains:")
    print(f"  MoM Adjustment (gamma^2):")
    print(f"    Proxy Adjustment:      {df_k['avg_proxy_adj_gamma_sq'].mean():.6f}")
    print(f"    PPI Adjustment:        {df_k['avg_ppi_adj_gamma_sq'].mean():.6f}")
    print(f"    PPI + W Adjustment:    {df_k['avg_ppi_weighted_adj_gamma_sq'].mean():.6f}")
    print(f"  Domain Bootstrap (rho, gamma^2):")
    print(f"    Proxy Dom Boot rho:    {df_k['avg_proxy_domain_boot_rho'].mean():.6f}")
    print(f"    Proxy Dom Boot gamma^2:{df_k['avg_proxy_domain_boot_gamma_sq'].mean():.6f}")
    print(f"    PPI Dom Boot rho:      {df_k['avg_ppi_domain_boot_rho'].mean():.6f}")
    print(f"    PPI Dom Boot gamma^2:  {df_k['avg_ppi_domain_boot_gamma_sq'].mean():.6f}")

print("\n" + "="*80)
print(f"\nResults saved to: {OUTPUT_DIR}")
print("  - simulation_results_all.csv (individual simulation results)")
print("  - coverage_summary.csv (aggregated coverage by condition)")
print("  - true_prevalences.csv (true target prevalences for each kappa)")
print("  - coverage_*.png (coverage visualization plots)")
print("  - ci_length_*.png (CI length visualization plots)")
print("  - coverage_vs_ci_length.png (coverage vs precision trade-off)")

In [ ]:
# =============================================================================
# DEBUGGING CELL: Estimate Differences and Latent Model Parameters
# =============================================================================
# This cell runs a single simulation and computes:
# 1. Differences between the primary estimate and proxy, PPI, PPI-W estimates
# 2. The latent model parameters (rho and gamma) for each method

def debug_estimate_differences(
    kappa: float = 0.01,
    sample_size: int = 1000,
    n_domains: int = 5,
    seed: int = 42,
):
    """
    Run a single simulation and analyze estimate differences and latent parameters.

    Computes:
    - Differences between primary estimate and each of: Proxy, PPI, PPI-W
    - Latent model parameters (rho and gamma_squared) for Proxy+Adj, PPI+Adj, PPI-W+Adj

    Note: This does NOT include bootstrapped versions.
    """
    print("="*80)
    print("DEBUGGING: Estimate Differences and Latent Model Parameters")
    print("="*80)
    print(f"\nParameters:")
    print(f"  - kappa (concept drift): {kappa}")
    print(f"  - sample_size: {sample_size}")
    print(f"  - n_domains (K): {n_domains}")
    print(f"  - seed: {seed}")

    # Step 1: Generate data
    print("\n" + "-"*80)
    print("STEP 1: Data Generation")
    print("-"*80)

    generator = CovShiftDataGenerator(
        n_samples_per_domain=sample_size,
        n_domains=n_domains,
        concept_drift_degree=kappa,
        primary_lambda=PRIMARY_LAMBDA,
        primary_phi=PRIMARY_PHI,
        proxy_lambda=PROXY_LAMBDA,
        proxy_phi=PROXY_PHI,
        n_covariates=P,
        target_domain_means=mu_K,
        random_seed=seed,
    )
    data = generator.generate_data()

    print(f"  Generated {len(data)} total samples across {n_domains} domains")
    print(f"  Samples per domain: {data.groupby('domain').size().to_dict()}")

    # Compute true prevalence
    true_prev = generator.calculate_target_population_prevalence(n_samples=1_000_000)
    print(f"\n  True target prevalence: {true_prev:.6f}")

    # Step 2: Create estimator
    print("\n" + "-"*80)
    print("STEP 2: Create Estimator")
    print("-"*80)

    estimator = AdjustedCovShiftPPIEstimator(
        df=data,
        primary_outcome_column="primary_outcome",
        proxy_outcome_column="proxy_outcome",
        importance_weight_column="importance_weight",
        domain_column="domain",
        target_domain_value=n_domains,
    )
    print("  Created AdjustedCovShiftPPIEstimator")

    # Step 3: Compute base estimates (Primary, Proxy, PPI, PPI-W)
    print("\n" + "-"*80)
    print("STEP 3: Compute Base Estimates")
    print("-"*80)

    # Primary (Oracle)
    primary_est = estimator.compute_primary_mean_estimate(confidence_level=CONFIDENCE_LEVEL)
    print(f"\n  Primary (Oracle):")
    print(f"     Estimate: {primary_est.estimate_val:.6f}")
    print(f"     95% CI: [{primary_est.lower_bound:.6f}, {primary_est.upper_bound:.6f}]")

    # Proxy Only
    proxy_est = estimator.compute_proxy_mean_estimate(confidence_level=CONFIDENCE_LEVEL)
    print(f"\n  Proxy Only:")
    print(f"     Estimate: {proxy_est.estimate_val:.6f}")
    print(f"     95% CI: [{proxy_est.lower_bound:.6f}, {proxy_est.upper_bound:.6f}]")

    # PPI (unweighted)
    ppi_est = estimator.compute_ppi_mean_estimate(weighted=False, confidence_level=CONFIDENCE_LEVEL)
    print(f"\n  PPI (unweighted):")
    print(f"     Estimate: {ppi_est.estimate_val:.6f}")
    print(f"     95% CI: [{ppi_est.lower_bound:.6f}, {ppi_est.upper_bound:.6f}]")

    # PPI-W (weighted)
    ppi_w_est = estimator.compute_ppi_mean_estimate(weighted=True, confidence_level=CONFIDENCE_LEVEL)
    print(f"\n  PPI-W (weighted):")
    print(f"     Estimate: {ppi_w_est.estimate_val:.6f}")
    print(f"     95% CI: [{ppi_w_est.lower_bound:.6f}, {ppi_w_est.upper_bound:.6f}]")

    # Step 4: Compute differences from primary estimate
    print("\n" + "-"*80)
    print("STEP 4: Differences from Primary Estimate")
    print("-"*80)

    diff_proxy = proxy_est.estimate_val - primary_est.estimate_val
    diff_ppi = ppi_est.estimate_val - primary_est.estimate_val
    diff_ppi_w = ppi_w_est.estimate_val - primary_est.estimate_val

    print(f"\n  Primary Estimate (reference): {primary_est.estimate_val:.6f}")
    print(f"\n  Differences (Method - Primary):")
    print(f"     Proxy - Primary:  {diff_proxy:+.6f}")
    print(f"     PPI - Primary:    {diff_ppi:+.6f}")
    print(f"     PPI-W - Primary:  {diff_ppi_w:+.6f}")

    # Step 5: Compute adjusted estimates and get latent model parameters
    print("\n" + "-"*80)
    print("STEP 5: Latent Model Parameters (rho and gamma) from MoM Adjustment")
    print("-"*80)
    print("\n  Note: These are from the non-bootstrapped MoM-adjusted estimates.")

    # Proxy + Adjustment (MoM)
    proxy_adj_est = estimator.compute_adjusted_proxy_estimate(
        confidence_level=CONFIDENCE_LEVEL, method="MoM"
    )
    print(f"\n  Proxy + Adjustment (MoM):")
    print(f"     Adjusted Estimate: {proxy_adj_est.estimate_val:.6f}")
    print(f"     rho (bias correction):     {proxy_adj_est.rho:.6f}")
    print(f"     gamma^2 (variance inflation): {proxy_adj_est.gamma_squared:.6f}")

    # PPI + Adjustment (MoM)
    ppi_adj_est = estimator.compute_adjusted_ppi_estimate(
        weighted=False, confidence_level=CONFIDENCE_LEVEL, method="MoM"
    )
    print(f"\n  PPI + Adjustment (MoM):")
    print(f"     Adjusted Estimate: {ppi_adj_est.estimate_val:.6f}")
    print(f"     rho (bias correction):     {ppi_adj_est.rho:.6f}")
    print(f"     gamma^2 (variance inflation): {ppi_adj_est.gamma_squared:.6f}")

    # PPI-W + Adjustment (MoM)
    ppi_w_adj_est = estimator.compute_adjusted_ppi_estimate(
        weighted=True, confidence_level=CONFIDENCE_LEVEL, method="MoM"
    )
    print(f"\n  PPI-W + Adjustment (MoM):")
    print(f"     Adjusted Estimate: {ppi_w_adj_est.estimate_val:.6f}")
    print(f"     rho (bias correction):     {ppi_w_adj_est.rho:.6f}")
    print(f"     gamma^2 (variance inflation): {ppi_w_adj_est.gamma_squared:.6f}")

    # Step 6: Summary Table
    print("\n" + "-"*80)
    print("STEP 6: Summary Table")
    print("-"*80)

    print(f"\n{'Method':<20} {'Estimate':>12} {'Diff from Primary':>18} {'rho':>12} {'gamma^2':>12}")
    print("-" * 78)

    # Primary (no latent params)
    print(f"{'Primary (Oracle)':<20} {primary_est.estimate_val:>12.6f} {'(reference)':>18} {'-':>12} {'-':>12}")

    # Proxy (base + adjusted)
    print(f"{'Proxy (base)':<20} {proxy_est.estimate_val:>12.6f} {diff_proxy:>+18.6f} {'-':>12} {'-':>12}")
    print(f"{'Proxy + Adj':<20} {proxy_adj_est.estimate_val:>12.6f} {proxy_adj_est.estimate_val - primary_est.estimate_val:>+18.6f} {proxy_adj_est.rho:>12.6f} {proxy_adj_est.gamma_squared:>12.6f}")

    # PPI (base + adjusted)
    print(f"{'PPI (base)':<20} {ppi_est.estimate_val:>12.6f} {diff_ppi:>+18.6f} {'-':>12} {'-':>12}")
    print(f"{'PPI + Adj':<20} {ppi_adj_est.estimate_val:>12.6f} {ppi_adj_est.estimate_val - primary_est.estimate_val:>+18.6f} {ppi_adj_est.rho:>12.6f} {ppi_adj_est.gamma_squared:>12.6f}")

    # PPI-W (base + adjusted)
    print(f"{'PPI-W (base)':<20} {ppi_w_est.estimate_val:>12.6f} {diff_ppi_w:>+18.6f} {'-':>12} {'-':>12}")
    print(f"{'PPI-W + Adj':<20} {ppi_w_adj_est.estimate_val:>12.6f} {ppi_w_adj_est.estimate_val - primary_est.estimate_val:>+18.6f} {ppi_w_adj_est.rho:>12.6f} {ppi_w_adj_est.gamma_squared:>12.6f}")

    print(f"\nTrue prevalence: {true_prev:.6f}")

    # Return results as a dictionary for further analysis
    results = {
        "primary": {
            "estimate": primary_est.estimate_val,
            "lower": primary_est.lower_bound,
            "upper": primary_est.upper_bound,
        },
        "proxy": {
            "estimate": proxy_est.estimate_val,
            "diff_from_primary": diff_proxy,
            "adjusted_estimate": proxy_adj_est.estimate_val,
            "rho": proxy_adj_est.rho,
            "gamma_squared": proxy_adj_est.gamma_squared,
        },
        "ppi": {
            "estimate": ppi_est.estimate_val,
            "diff_from_primary": diff_ppi,
            "adjusted_estimate": ppi_adj_est.estimate_val,
            "rho": ppi_adj_est.rho,
            "gamma_squared": ppi_adj_est.gamma_squared,
        },
        "ppi_w": {
            "estimate": ppi_w_est.estimate_val,
            "diff_from_primary": diff_ppi_w,
            "adjusted_estimate": ppi_w_adj_est.estimate_val,
            "rho": ppi_w_adj_est.rho,
            "gamma_squared": ppi_w_adj_est.gamma_squared,
        },
        "true_prevalence": true_prev,
    }

    return results


# Run the debugging analysis
diff_debug_results = debug_estimate_differences(
    kappa=0.01,
    sample_size=1000,
    n_domains=5,
    seed=42,
)

In [ ]:
kappa: float = 0.01
sample_size: int = 1000
n_domains: int = 5
seed: int = 42


print("="*80)
print("DEBUGGING: Estimate Differences and Latent Model Parameters")
print("="*80)
print(f"\nParameters:")
print(f"  - kappa (concept drift): {kappa}")
print(f"  - sample_size: {sample_size}")
print(f"  - n_domains (K): {n_domains}")
print(f"  - seed: {seed}")

# Step 1: Generate data
print("\n" + "-"*80)
print("STEP 1: Data Generation")
print("-"*80)

generator = CovShiftDataGenerator(
    n_samples_per_domain=sample_size,
    n_domains=n_domains,
    concept_drift_degree=kappa,
    primary_lambda=PRIMARY_LAMBDA,
    primary_phi=PRIMARY_PHI,
    proxy_lambda=PROXY_LAMBDA,
    proxy_phi=PROXY_PHI,
    n_covariates=P,
    target_domain_means=mu_K,
    random_seed=seed,
)
data = generator.generate_data()

print(f"  Generated {len(data)} total samples across {n_domains} domains")
print(f"  Samples per domain: {data.groupby('domain').size().to_dict()}")

# Compute true prevalence
true_prev = generator.calculate_target_population_prevalence(n_samples=1_000_000)
print(f"\n  True target prevalence: {true_prev:.6f}")

# Step 2: Create estimator
print("\n" + "-"*80)
print("STEP 2: Create Estimator")
print("-"*80)

estimator = AdjustedCovShiftPPIEstimator(
    df=data,
    primary_outcome_column="primary_outcome",
    proxy_outcome_column="proxy_outcome",
    importance_weight_column="importance_weight",
    domain_column="domain",
    target_domain_value=n_domains,
    cross_domain_weight_pattern = "weight_to_domain_{target_domain}",
)
print("  Created AdjustedCovShiftPPIEstimator")

# Step 3: Compute base estimates (Primary, Proxy, PPI, PPI-W)
print("\n" + "-"*80)
print("STEP 3: Compute Base Estimates")
print("-"*80)

# Primary (Oracle)
primary_est = estimator.compute_primary_mean_estimate(confidence_level=CONFIDENCE_LEVEL)
print(f"\n  Primary (Oracle):")
print(f"     Estimate: {primary_est.estimate_val:.6f}")
print(f"     95% CI: [{primary_est.lower_bound:.6f}, {primary_est.upper_bound:.6f}]")

# Proxy Only
proxy_est = estimator.compute_proxy_mean_estimate(confidence_level=CONFIDENCE_LEVEL)
print(f"\n  Proxy Only:")
print(f"     Estimate: {proxy_est.estimate_val:.6f}")
print(f"     95% CI: [{proxy_est.lower_bound:.6f}, {proxy_est.upper_bound:.6f}]")


In [ ]:
# PPI (unweighted)
ppi_est = estimator.compute_ppi_mean_estimate(weighted=False, confidence_level=CONFIDENCE_LEVEL)
print(f"\n  PPI (unweighted):")
print(f"     Estimate: {ppi_est.estimate_val:.6f}")
print(f"     95% CI: [{ppi_est.lower_bound:.6f}, {ppi_est.upper_bound:.6f}]")


In [ ]:
# PPI-W (weighted)
ppi_w_est = estimator.compute_ppi_mean_estimate(weighted=True, confidence_level=CONFIDENCE_LEVEL)
print(f"\n  PPI-W (weighted):")
print(f"     Estimate: {ppi_w_est.estimate_val:.6f}")
print(f"     95% CI: [{ppi_w_est.lower_bound:.6f}, {ppi_w_est.upper_bound:.6f}]")

# Step 4: Compute differences from primary estimate
print("\n" + "-"*80)
print("STEP 4: Differences from Primary Estimate")
print("-"*80)


diff_proxy = proxy_est.estimate_val - primary_est.estimate_val
diff_ppi = ppi_est.estimate_val - primary_est.estimate_val
diff_ppi_w = ppi_w_est.estimate_val - primary_est.estimate_val

print(f"\n  Primary Estimate (reference): {primary_est.estimate_val:.6f}")
print(f"\n  Differences (Method - Primary):")
print(f"     Proxy - Primary:  {diff_proxy:+.6f}")
print(f"     PPI - Primary:    {diff_ppi:+.6f}")
print(f"     PPI-W - Primary:  {diff_ppi_w:+.6f}")

In [ ]:
# Step 5: Compute adjusted estimates and get latent model parameters
print("\n" + "-"*80)
print("STEP 5: Latent Model Parameters (rho and gamma) from MoM Adjustment")
print("-"*80)
print("\n  Note: These are from the non-bootstrapped MoM-adjusted estimates.")

# Proxy + Adjustment (MoM)
proxy_adj_est = estimator.compute_adjusted_proxy_estimate(
    confidence_level=CONFIDENCE_LEVEL, method="MoM"
)
print(f"\n  Proxy + Adjustment (MoM):")
print(f"     Adjusted Estimate: {proxy_adj_est.estimate_val:.6f}")
print(f"     rho (bias correction):     {proxy_adj_est.rho:.6f}")
print(f"     gamma^2 (variance inflation): {proxy_adj_est.gamma_squared:.6f}")

# PPI + Adjustment (MoM)
ppi_adj_est = estimator.compute_adjusted_ppi_estimate(
    weighted=False, confidence_level=CONFIDENCE_LEVEL, method="MoM"
)
print(f"\n  PPI + Adjustment (MoM):")
print(f"     Adjusted Estimate: {ppi_adj_est.estimate_val:.6f}")
print(f"     rho (bias correction):     {ppi_adj_est.rho:.6f}")
print(f"     gamma^2 (variance inflation): {ppi_adj_est.gamma_squared:.6f}")

# PPI-W + Adjustment (MoM)
ppi_w_adj_est = estimator.compute_adjusted_ppi_estimate(
    weighted=True, confidence_level=CONFIDENCE_LEVEL, method="MoM"
)
print(f"\n  PPI-W + Adjustment (MoM):")
print(f"     Adjusted Estimate: {ppi_w_adj_est.estimate_val:.6f}")
print(f"     rho (bias correction):     {ppi_w_adj_est.rho:.6f}")
print(f"     gamma^2 (variance inflation): {ppi_w_adj_est.gamma_squared:.6f}")

# Step 6: Summary Table
print("\n" + "-"*80)
print("STEP 6: Summary Table")
print("-"*80)

print(f"\n{'Method':<20} {'Estimate':>12} {'Diff from Primary':>18} {'rho':>12} {'gamma^2':>12}")
print("-" * 78)

# Primary (no latent params)
print(f"{'Primary (Oracle)':<20} {primary_est.estimate_val:>12.6f} {'(reference)':>18} {'-':>12} {'-':>12}")

# Proxy (base + adjusted)
print(f"{'Proxy (base)':<20} {proxy_est.estimate_val:>12.6f} {diff_proxy:>+18.6f} {'-':>12} {'-':>12}")
print(f"{'Proxy + Adj':<20} {proxy_adj_est.estimate_val:>12.6f} {proxy_adj_est.estimate_val - primary_est.estimate_val:>+18.6f} {proxy_adj_est.rho:>12.6f} {proxy_adj_est.gamma_squared:>12.6f}")

# PPI (base + adjusted)
print(f"{'PPI (base)':<20} {ppi_est.estimate_val:>12.6f} {diff_ppi:>+18.6f} {'-':>12} {'-':>12}")
print(f"{'PPI + Adj':<20} {ppi_adj_est.estimate_val:>12.6f} {ppi_adj_est.estimate_val - primary_est.estimate_val:>+18.6f} {ppi_adj_est.rho:>12.6f} {ppi_adj_est.gamma_squared:>12.6f}")

# PPI-W (base + adjusted)
print(f"{'PPI-W (base)':<20} {ppi_w_est.estimate_val:>12.6f} {diff_ppi_w:>+18.6f} {'-':>12} {'-':>12}")
print(f"{'PPI-W + Adj':<20} {ppi_w_adj_est.estimate_val:>12.6f} {ppi_w_adj_est.estimate_val - primary_est.estimate_val:>+18.6f} {ppi_w_adj_est.rho:>12.6f} {ppi_w_adj_est.gamma_squared:>12.6f}")

print(f"\nTrue prevalence: {true_prev:.6f}")

# Return results as a dictionary for further analysis
results = {
    "primary": {
        "estimate": primary_est.estimate_val,
        "lower": primary_est.lower_bound,
        "upper": primary_est.upper_bound,
    },
    "proxy": {
        "estimate": proxy_est.estimate_val,
        "diff_from_primary": diff_proxy,
        "adjusted_estimate": proxy_adj_est.estimate_val,
        "rho": proxy_adj_est.rho,
        "gamma_squared": proxy_adj_est.gamma_squared,
    },
    "ppi": {
        "estimate": ppi_est.estimate_val,
        "diff_from_primary": diff_ppi,
        "adjusted_estimate": ppi_adj_est.estimate_val,
        "rho": ppi_adj_est.rho,
        "gamma_squared": ppi_adj_est.gamma_squared,
    },
    "ppi_w": {
        "estimate": ppi_w_est.estimate_val,
        "diff_from_primary": diff_ppi_w,
        "adjusted_estimate": ppi_w_adj_est.estimate_val,
        "rho": ppi_w_adj_est.rho,
        "gamma_squared": ppi_w_adj_est.gamma_squared,
    },
    "true_prevalence": true_prev,
}


In [ ]:
weighted: bool = True
confidence_level: float = 0.95
method: str = "MoM"

"""
Compute PPI estimate with adjusted confidence intervals.

The adjusted SE accounts for cross-domain heterogeneity:
    adjusted_se = sqrt(base_se^2 + gamma^2)

Args:
    weighted: If True, use importance weights.
    confidence_level: Confidence level for intervals.
    method: Latent model estimation method ("MoM" or "MLE").

Returns:
    AdjustedEstimate with original and adjusted bounds.
"""
base_estimate = estimator.compute_ppi_mean_estimate(
    weighted=weighted,
    confidence_level=confidence_level,
)

fit_result = estimator.estimate_latent_parameters(
    use_ppi_errors=True,
    weighted=weighted,
    method=method,
)
gamma_squared = fit_result.gamma_squared
rho = fit_result.rho

total_variance = base_estimate.standard_error**2 + gamma_squared
adjusted_est = base_estimate.estimate_val + rho
adjusted_se = float(np.sqrt(total_variance))

z = norm.ppf(1 - (1 - confidence_level) / 2)
adjusted_lower = adjusted_est - z * adjusted_se
adjusted_upper = adjusted_est + z * adjusted_se

In [ ]:
Adjusted_Estimator = AdjustedCovShiftPPIEstimator(
    df=data,
    primary_outcome_column="primary_outcome",
    proxy_outcome_column="proxy_outcome",
    importance_weight_column="importance_weight",
    domain_column="domain",
    target_domain_value=n_domains,
    cross_domain_weight_pattern='weight_to_domain_{target_domain}'
)

In [ ]:
# =============================================================================
# DEBUGGING CELL: Primary vs Proxy Differences and PPI Error Visualization
# =============================================================================
# This cell runs a single simulation and visualizes:
# 1. Differences between primary and proxy estimates across domains (with CIs)
# 2. The PPI error values from hold_one_domain_out_estimate_stats
# 3. Latent model parameters (rho and gamma) for each method

import matplotlib.pyplot as plt
from scipy.stats import norm

def debug_primary_proxy_differences(
    kappa: float = 0.01,
    sample_size: int = 1000,
    n_domains: int = 5,
    seed: int = 42,
):
    """
    Run a single simulation and visualize primary vs proxy differences and PPI error.

    This debugging cell:
    1. Generates simulation data
    2. Creates AdjustedCovShiftPPIEstimator with cross_domain_weight_pattern
    3. Computes hold_one_domain_out_estimate_stats to get per-domain statistics
    4. Plots differences between primary and proxy estimates (with confidence intervals)
    5. Visualizes the PPI error values from the estimator
    """
    print("="*80)
    print("DEBUGGING: Primary vs Proxy Differences and PPI Error Visualization")
    print("="*80)
    print(f"\nParameters:")
    print(f"  - kappa (concept drift): {kappa}")
    print(f"  - sample_size: {sample_size}")
    print(f"  - n_domains (K): {n_domains}")
    print(f"  - seed: {seed}")

    # Step 1: Generate data
    print("\n" + "-"*80)
    print("STEP 1: Data Generation")
    print("-"*80)

    generator = CovShiftDataGenerator(
        n_samples_per_domain=sample_size,
        n_domains=n_domains,
        concept_drift_degree=kappa,
        primary_lambda=PRIMARY_LAMBDA,
        primary_phi=PRIMARY_PHI,
        proxy_lambda=PROXY_LAMBDA,
        proxy_phi=PROXY_PHI,
        n_covariates=P,
        target_domain_means=mu_K,
        random_seed=seed,
    )
    data = generator.generate_data()

    print(f"  Generated {len(data)} total samples across {n_domains} domains")
    print(f"  Samples per domain: {data.groupby('domain').size().to_dict()}")

    # Compute true prevalence
    true_prev = generator.calculate_target_population_prevalence(n_samples=1_000_000)
    print(f"\n  True target prevalence: {true_prev:.6f}")

    # Step 2: Create estimator with cross_domain_weight_pattern
    print("\n" + "-"*80)
    print("STEP 2: Create Estimator with Cross-Domain Weight Pattern")
    print("-"*80)

    Adjusted_Estimator = AdjustedCovShiftPPIEstimator(
        df=data,
        primary_outcome_column="primary_outcome",
        proxy_outcome_column="proxy_outcome",
        importance_weight_column="importance_weight",
        domain_column="domain",
        target_domain_value=n_domains,
        cross_domain_weight_pattern='weight_to_domain_{target_domain}'
    )
    print("  Created AdjustedCovShiftPPIEstimator with cross_domain_weight_pattern")

    # Step 3: Compute hold-one-domain-out estimate stats
    print("\n" + "-"*80)
    print("STEP 3: Compute Hold-One-Domain-Out Estimate Stats")
    print("-"*80)

    # Compute for both weighted and unweighted
    domain_stats_unweighted = Adjusted_Estimator.compute_hold_one_domain_out_estimate_stats(weighted=False)
    domain_stats_weighted = Adjusted_Estimator.compute_hold_one_domain_out_estimate_stats(weighted=True)

    print(f"\n  Computed stats for {len(domain_stats_unweighted)} source domains (unweighted)")
    print(f"  Computed stats for {len(domain_stats_weighted)} source domains (weighted)")

    # Step 4: Print domain statistics table
    print("\n" + "-"*80)
    print("STEP 4: Per-Domain Statistics")
    print("-"*80)

    print(f"\n  {'Domain':<8} {'N':<6} {'Proxy':>10} {'Primary':>10} {'Diff':>10} {'Rectifier':>12} {'Rect Proxy':>12} {'PPI Error':>10}")
    print("  " + "-" * 90)

    for stats in domain_stats_weighted:
        rectifier_str = f"{stats.rectifier:.6f}" if stats.rectifier is not None else "N/A"
        rect_proxy_str = f"{stats.rectified_proxy_mean:.6f}" if stats.rectified_proxy_mean is not None else "N/A"
        ppi_error_str = f"{stats.ppi_error:.6f}" if stats.ppi_error is not None else "N/A"
        print(f"  {stats.domain:<8} {stats.n_samples:<6} {stats.proxy_mean:>10.6f} {stats.primary_mean:>10.6f} "
              f"{stats.est_diff:>10.6f} {rectifier_str:>12} {rect_proxy_str:>12} {ppi_error_str:>10}")

    # Step 5: Compute base and adjusted estimates
    print("\n" + "-"*80)
    print("STEP 5: Compute Estimates and Latent Parameters")
    print("-"*80)

    # Primary (Oracle)
    primary_est = Adjusted_Estimator.compute_primary_mean_estimate(confidence_level=CONFIDENCE_LEVEL)
    print(f"\n  Primary (Oracle): {primary_est.estimate_val:.6f}")

    # Proxy Only
    proxy_est = Adjusted_Estimator.compute_proxy_mean_estimate(confidence_level=CONFIDENCE_LEVEL)
    print(f"  Proxy Only: {proxy_est.estimate_val:.6f}")

    # PPI (unweighted)
    ppi_est = Adjusted_Estimator.compute_ppi_mean_estimate(weighted=False, confidence_level=CONFIDENCE_LEVEL)
    print(f"  PPI (unweighted): {ppi_est.estimate_val:.6f}")

    # PPI-W (weighted)
    ppi_w_est = Adjusted_Estimator.compute_ppi_mean_estimate(weighted=True, confidence_level=CONFIDENCE_LEVEL)
    print(f"  PPI-W (weighted): {ppi_w_est.estimate_val:.6f}")

    # Adjusted estimates with latent parameters
    proxy_adj_est = Adjusted_Estimator.compute_adjusted_proxy_estimate(
        confidence_level=CONFIDENCE_LEVEL, method="MoM"
    )
    ppi_adj_est = Adjusted_Estimator.compute_adjusted_ppi_estimate(
        weighted=False, confidence_level=CONFIDENCE_LEVEL, method="MoM"
    )
    ppi_w_adj_est = Adjusted_Estimator.compute_adjusted_ppi_estimate(
        weighted=True, confidence_level=CONFIDENCE_LEVEL, method="MoM"
    )

    print(f"\n  Latent Model Parameters (MoM):")
    print(f"    Proxy+Adj:  rho = {proxy_adj_est.rho:+.6f}, gamma^2 = {proxy_adj_est.gamma_squared:.6f}")
    print(f"    PPI+Adj:    rho = {ppi_adj_est.rho:+.6f}, gamma^2 = {ppi_adj_est.gamma_squared:.6f}")
    print(f"    PPI-W+Adj:  rho = {ppi_w_adj_est.rho:+.6f}, gamma^2 = {ppi_w_adj_est.gamma_squared:.6f}")

    # Step 6: Create visualizations (points with error bars, domains on Y-axis)
    print("\n" + "-"*80)
    print("STEP 6: Visualizations")
    print("-"*80)

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    # Extract data for plotting
    domains = [s.domain for s in domain_stats_weighted]
    proxy_means = np.array([s.proxy_mean for s in domain_stats_weighted])
    proxy_ses = np.array([s.proxy_se for s in domain_stats_weighted])
    primary_means = np.array([s.primary_mean for s in domain_stats_weighted])
    primary_ses = np.array([s.primary_se for s in domain_stats_weighted])
    est_diffs = np.array([s.est_diff for s in domain_stats_weighted])
    est_diff_vars = np.array([s.est_diff_var for s in domain_stats_weighted])
    ppi_errors = np.array([s.ppi_error if s.ppi_error is not None else 0 for s in domain_stats_weighted])
    ppi_error_vars = np.array([s.ppi_error_var if s.ppi_error_var is not None else 0 for s in domain_stats_weighted])
    rectified_proxies = np.array([s.rectified_proxy_mean if s.rectified_proxy_mean is not None else s.proxy_mean for s in domain_stats_weighted])

    # Compute 95% CI half-widths (z = 1.96 for 95%)
    z = norm.ppf(0.975)
    proxy_ci = z * proxy_ses
    primary_ci = z * primary_ses
    diff_ci = z * np.sqrt(est_diff_vars)
    ppi_error_ci = z * np.sqrt(ppi_error_vars)

    domain_labels = [f'Domain {d}' for d in domains]
    y_pos = np.arange(len(domains))

    # Plot 1: Primary vs Proxy estimates by domain (points with error bars)
    ax1 = axes[0, 0]
    offset = 0.1
    ax1.errorbar(proxy_means, y_pos + offset, xerr=proxy_ci, fmt='o',
                 label='Proxy Mean', color='steelblue', markersize=8, capsize=4, capthick=2)
    ax1.errorbar(primary_means, y_pos - offset, xerr=primary_ci, fmt='s',
                 label='Primary Mean', color='darkorange', markersize=8, capsize=4, capthick=2)
    ax1.axvline(x=true_prev, color='red', linestyle='--', linewidth=2, label=f'True Prevalence ({true_prev:.4f})')
    ax1.set_ylabel('Domain')
    ax1.set_xlabel('Estimate')
    ax1.set_title('Primary vs Proxy Estimates by Domain (with 95% CI)')
    ax1.set_yticks(y_pos)
    ax1.set_yticklabels(domain_labels)
    ax1.legend(loc='best')
    ax1.grid(axis='x', alpha=0.3)

    # Plot 2: Difference (Primary - Proxy) by domain (points with error bars)
    ax2 = axes[0, 1]
    colors = ['green' if d > 0 else 'red' for d in est_diffs]
    for i, (diff, ci, color) in enumerate(zip(est_diffs, diff_ci, colors)):
        ax2.errorbar(diff, y_pos[i], xerr=ci, fmt='o', color=color, markersize=8, capsize=4, capthick=2)
    ax2.axvline(x=0, color='black', linestyle='-', linewidth=1)
    ax2.axvline(x=np.mean(est_diffs), color='blue', linestyle='--', linewidth=2,
                label=f'Mean Diff ({np.mean(est_diffs):.4f})')
    ax2.set_ylabel('Domain')
    ax2.set_xlabel('Difference (Primary - Proxy)')
    ax2.set_title('Primary - Proxy Difference by Domain (with 95% CI)')
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(domain_labels)
    ax2.legend(loc='best')
    ax2.grid(axis='x', alpha=0.3)

    # Plot 3: PPI Error by domain (points with error bars)
    ax3 = axes[1, 0]
    colors_ppi = ['green' if e > 0 else 'red' for e in ppi_errors]
    for i, (ppi_err, ci, color) in enumerate(zip(ppi_errors, ppi_error_ci, colors_ppi)):
        ax3.errorbar(ppi_err, y_pos[i], xerr=ci, fmt='o', color=color, markersize=8, capsize=4, capthick=2)
    ax3.axvline(x=0, color='black', linestyle='-', linewidth=1)
    ax3.axvline(x=np.mean(ppi_errors), color='blue', linestyle='--', linewidth=2,
                label=f'Mean PPI Error ({np.mean(ppi_errors):.4f})')
    ax3.set_ylabel('Domain')
    ax3.set_xlabel('PPI Error (Primary - Rectified Proxy)')
    ax3.set_title('PPI Error by Domain (with 95% CI)')
    ax3.set_yticks(y_pos)
    ax3.set_yticklabels(domain_labels)
    ax3.legend(loc='best')
    ax3.grid(axis='x', alpha=0.3)

    # Plot 4: Comparison of Proxy, Rectified Proxy, and Primary (points with error bars)
    ax4 = axes[1, 1]
    offset = 0.15
    ax4.errorbar(proxy_means, y_pos + offset, xerr=proxy_ci, fmt='o',
                 label='Proxy', color='steelblue', markersize=8, capsize=4, capthick=2)
    ax4.plot(rectified_proxies, y_pos, 'd', label='Rectified Proxy', color='purple', markersize=10)
    ax4.errorbar(primary_means, y_pos - offset, xerr=primary_ci, fmt='s',
                 label='Primary', color='darkorange', markersize=8, capsize=4, capthick=2)
    ax4.axvline(x=true_prev, color='red', linestyle='--', linewidth=2, label='True Prevalence')
    ax4.set_ylabel('Domain')
    ax4.set_xlabel('Estimate')
    ax4.set_title('Proxy vs Rectified Proxy vs Primary (with 95% CI)')
    ax4.set_yticks(y_pos)
    ax4.set_yticklabels(domain_labels)
    ax4.legend(loc='best')
    ax4.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Step 7: Summary table
    print("\n" + "-"*80)
    print("STEP 7: Summary Table - Differences from Primary")
    print("-"*80)

    diff_proxy = proxy_est.estimate_val - primary_est.estimate_val
    diff_ppi = ppi_est.estimate_val - primary_est.estimate_val
    diff_ppi_w = ppi_w_est.estimate_val - primary_est.estimate_val
    diff_proxy_adj = proxy_adj_est.estimate_val - primary_est.estimate_val
    diff_ppi_adj = ppi_adj_est.estimate_val - primary_est.estimate_val
    diff_ppi_w_adj = ppi_w_adj_est.estimate_val - primary_est.estimate_val

    print(f"\n  {'Method':<20} {'Estimate':>12} {'Diff from Primary':>18} {'rho':>12} {'gamma^2':>12}")
    print("  " + "-" * 76)
    print(f"  {'Primary (Oracle)':<20} {primary_est.estimate_val:>12.6f} {'(reference)':>18} {'-':>12} {'-':>12}")
    print(f"  {'Proxy':<20} {proxy_est.estimate_val:>12.6f} {diff_proxy:>+18.6f} {'-':>12} {'-':>12}")
    print(f"  {'Proxy + Adj':<20} {proxy_adj_est.estimate_val:>12.6f} {diff_proxy_adj:>+18.6f} {proxy_adj_est.rho:>+12.6f} {proxy_adj_est.gamma_squared:>12.6f}")
    print(f"  {'PPI':<20} {ppi_est.estimate_val:>12.6f} {diff_ppi:>+18.6f} {'-':>12} {'-':>12}")
    print(f"  {'PPI + Adj':<20} {ppi_adj_est.estimate_val:>12.6f} {diff_ppi_adj:>+18.6f} {ppi_adj_est.rho:>+12.6f} {ppi_adj_est.gamma_squared:>12.6f}")
    print(f"  {'PPI-W':<20} {ppi_w_est.estimate_val:>12.6f} {diff_ppi_w:>+18.6f} {'-':>12} {'-':>12}")
    print(f"  {'PPI-W + Adj':<20} {ppi_w_adj_est.estimate_val:>12.6f} {diff_ppi_w_adj:>+18.6f} {ppi_w_adj_est.rho:>+12.6f} {ppi_w_adj_est.gamma_squared:>12.6f}")

    print(f"\n  True prevalence: {true_prev:.6f}")

    # Return results for further analysis
    results = {
        "domain_stats_weighted": domain_stats_weighted,
        "domain_stats_unweighted": domain_stats_unweighted,
        "estimates": {
            "primary": primary_est.estimate_val,
            "proxy": proxy_est.estimate_val,
            "ppi": ppi_est.estimate_val,
            "ppi_w": ppi_w_est.estimate_val,
            "proxy_adj": proxy_adj_est.estimate_val,
            "ppi_adj": ppi_adj_est.estimate_val,
            "ppi_w_adj": ppi_w_adj_est.estimate_val,
        },
        "latent_params": {
            "proxy_adj": {"rho": proxy_adj_est.rho, "gamma_squared": proxy_adj_est.gamma_squared},
            "ppi_adj": {"rho": ppi_adj_est.rho, "gamma_squared": ppi_adj_est.gamma_squared},
            "ppi_w_adj": {"rho": ppi_w_adj_est.rho, "gamma_squared": ppi_w_adj_est.gamma_squared},
        },
        "true_prevalence": true_prev,
    }

    return results


# Run the debugging analysis
debug_results = debug_primary_proxy_differences(
    kappa=0.00,
    sample_size=100000,
    n_domains=15,
    seed=43,
)


In [ ]:
kappa: float = 0.00
sample_size: int = 100000
n_domains: int = 15
seed: int = 44

generator = CovShiftDataGenerator(
    n_samples_per_domain=sample_size,
    n_domains=n_domains,
    concept_drift_degree=kappa,
    primary_lambda=PRIMARY_LAMBDA,
    primary_phi=PRIMARY_PHI,
    proxy_lambda=PROXY_LAMBDA,
    proxy_phi=PROXY_PHI,
    n_covariates=P,
    target_domain_means=mu_K,
    random_seed=seed,
)
data = generator.generate_data()

print(f"  Generated {len(data)} total samples across {n_domains} domains")
print(f"  Samples per domain: {data.groupby('domain').size().to_dict()}")

# Compute true prevalence
true_prev = generator.calculate_target_population_prevalence(n_samples=1_000_000)
print(f"\n  True target prevalence: {true_prev:.6f}")

# Step 2: Create estimator with cross_domain_weight_pattern
print("\n" + "-"*80)
print("STEP 2: Create Estimator with Cross-Domain Weight Pattern")
print("-"*80)

Adjusted_Estimator = AdjustedCovShiftPPIEstimator(
    df=data,
    primary_outcome_column="primary_outcome",
    proxy_outcome_column="proxy_outcome",
    importance_weight_column="importance_weight",
    domain_column="domain",
    target_domain_value=n_domains,
    cross_domain_weight_pattern='weight_to_domain_{target_domain}'
)
print("  Created AdjustedCovShiftPPIEstimator with cross_domain_weight_pattern")

# Step 3: Compute hold-one-domain-out estimate stats
print("\n" + "-"*80)
print("STEP 3: Compute Hold-One-Domain-Out Estimate Stats")
print("-"*80)

# Compute for both weighted and unweighted
domain_stats_unweighted = Adjusted_Estimator.compute_hold_one_domain_out_estimate_stats(weighted=False)
domain_stats_weighted = Adjusted_Estimator.compute_hold_one_domain_out_estimate_stats(weighted=True)

print(f"\n  Computed stats for {len(domain_stats_unweighted)} source domains (unweighted)")
print(f"  Computed stats for {len(domain_stats_weighted)} source domains (weighted)")

# Step 4: Print domain statistics table
print("\n" + "-"*80)
print("STEP 4: Per-Domain Statistics")
print("-"*80)

print(f"\n  {'Domain':<8} {'N':<6} {'Proxy':>10} {'Primary':>10} {'Diff':>10} {'Rectifier':>12} {'Rect Proxy':>12} {'PPI Error':>10}")
print("  " + "-" * 90)

for stats in domain_stats_weighted:
    rectifier_str = f"{stats.rectifier:.6f}" if stats.rectifier is not None else "N/A"
    rect_proxy_str = f"{stats.rectified_proxy_mean:.6f}" if stats.rectified_proxy_mean is not None else "N/A"
    ppi_error_str = f"{stats.ppi_error:.6f}" if stats.ppi_error is not None else "N/A"
    print(f"  {stats.domain:<8} {stats.n_samples:<6} {stats.proxy_mean:>10.6f} {stats.primary_mean:>10.6f} "
            f"{stats.est_diff:>10.6f} {rectifier_str:>12} {rect_proxy_str:>12} {ppi_error_str:>10}")

# Step 5: Compute base and adjusted estimates
print("\n" + "-"*80)
print("STEP 5: Compute Estimates and Latent Parameters")
print("-"*80)

# Primary (Oracle)
primary_est = Adjusted_Estimator.compute_primary_mean_estimate(confidence_level=CONFIDENCE_LEVEL)
print(f"\n  Primary (Oracle): {primary_est.estimate_val:.6f}")

# Proxy Only
proxy_est = Adjusted_Estimator.compute_proxy_mean_estimate(confidence_level=CONFIDENCE_LEVEL)
print(f"  Proxy Only: {proxy_est.estimate_val:.6f}")

# PPI (unweighted)
ppi_est = Adjusted_Estimator.compute_ppi_mean_estimate(weighted=False, confidence_level=CONFIDENCE_LEVEL)
print(f"  PPI (unweighted): {ppi_est.estimate_val:.6f}")

# PPI-W (weighted)
ppi_w_est = Adjusted_Estimator.compute_ppi_mean_estimate(weighted=True, confidence_level=CONFIDENCE_LEVEL)
print(f"  PPI-W (weighted): {ppi_w_est.estimate_val:.6f}")

# Adjusted estimates with latent parameters
proxy_adj_est = Adjusted_Estimator.compute_adjusted_proxy_estimate(
    confidence_level=CONFIDENCE_LEVEL, method="MoM"
)
ppi_adj_est = Adjusted_Estimator.compute_adjusted_ppi_estimate(
    weighted=False, confidence_level=CONFIDENCE_LEVEL, method="MoM"
)
ppi_w_adj_est = Adjusted_Estimator.compute_adjusted_ppi_estimate(
    weighted=True, confidence_level=CONFIDENCE_LEVEL, method="MoM"
)

print(f"\n  Latent Model Parameters (MoM):")
print(f"    Proxy+Adj:  rho = {proxy_adj_est.rho:+.6f}, gamma^2 = {proxy_adj_est.gamma_squared:.6f}")
print(f"    PPI+Adj:    rho = {ppi_adj_est.rho:+.6f}, gamma^2 = {ppi_adj_est.gamma_squared:.6f}")
print(f"    PPI-W+Adj:  rho = {ppi_w_adj_est.rho:+.6f}, gamma^2 = {ppi_w_adj_est.gamma_squared:.6f}")

# Step 6: Create visualizations (points with error bars, domains on Y-axis)
print("\n" + "-"*80)
print("STEP 6: Visualizations")
print("-"*80)

# Extract data for plotting
domains = [s.domain for s in domain_stats_weighted]
proxy_means = np.array([s.proxy_mean for s in domain_stats_weighted])
proxy_ses = np.array([s.proxy_se for s in domain_stats_weighted])
primary_means = np.array([s.primary_mean for s in domain_stats_weighted])
primary_ses = np.array([s.primary_se for s in domain_stats_weighted])
est_diffs = np.array([s.est_diff for s in domain_stats_weighted])
est_diff_vars = np.array([s.est_diff_var for s in domain_stats_weighted])
ppi_errors = np.array([s.ppi_error if s.ppi_error is not None else 0 for s in domain_stats_weighted])
ppi_error_vars = np.array([s.ppi_error_var if s.ppi_error_var is not None else 0 for s in domain_stats_weighted])
rectified_proxies = np.array([s.rectified_proxy_mean if s.rectified_proxy_mean is not None else s.proxy_mean for s in domain_stats_weighted])


# Plot the sequence of ppi_errors with confidence intervals

def plot_ppi_errors_with_confidence_intervals(domains, ppi_errors, ppi_error_vars):
    # Calculate confidence intervals
    alpha = 0.05
    z = norm.ppf(1 - alpha / 2)
    ci_lower = ppi_errors - z * np.sqrt(ppi_error_vars)
    ci_upper = ppi_errors + z * np.sqrt(ppi_error_vars)

    # Plotting
    plt.figure(figsize=(12, 6))
    plt.errorbar(domains, ppi_errors, yerr=z * np.sqrt(ppi_error_vars), fmt='o', elinewidth=2, capsize=4, label='PPI Errors', color='b')
    plt.fill_between(domains, ci_lower, ci_upper, color='b', alpha=0.1)

    plt.title('PPI Errors with Confidence Intervals')
    plt.xlabel('Domain')
    plt.ylabel('PPI Error')
    plt.xticks(domains)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

# Call the plotting function
plot_ppi_errors_with_confidence_intervals(domains, ppi_errors, ppi_error_vars)

In [ ]:
domain_stats_weighted

In [ ]:
# Plot the sequence of ppi_errors with confidence intervals

def plot_ppi_errors_with_confidence_intervals(domains, ppi_errors, ppi_error_vars):
    # Calculate confidence intervals
    alpha = 0.05
    z = norm.ppf(1 - alpha / 2)
    ci_lower = ppi_errors - z * np.sqrt(ppi_error_vars)
    ci_upper = ppi_errors + z * np.sqrt(ppi_error_vars)

    # Plotting
    plt.figure(figsize=(12, 6))
    plt.errorbar(domains, ppi_errors, yerr=z * np.sqrt(ppi_error_vars), fmt='o', elinewidth=2, capsize=4, label='PPI Errors', color='b')
    plt.fill_between(domains, ci_lower, ci_upper, color='b', alpha=0.1)

    plt.title('PPI Errors with Confidence Intervals')
    plt.xlabel('Domain')
    plt.ylabel('PPI Error')
    plt.xticks(domains)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

# Call the plotting function
plot_ppi_errors_with_confidence_intervals(domains, ppi_errors, ppi_error_vars)

In [ ]:
Adjusted_Estimator = AdjustedCovShiftPPIEstimator(
    df=data,
    primary_outcome_column="primary_outcome",
    proxy_outcome_column="proxy_outcome",
    importance_weight_column="importance_weight",
    domain_column="domain",
    target_domain_value=n_domains,
    cross_domain_weight_pattern='weight_to_domain_{target_domain}'
)

In [ ]:
import numpy.typing as npt
@dataclass
class DomainEstimateStats:
    """Per-domain estimate statistics for latent model fitting."""
    domain: Any
    n_samples: int
    # Basic estimates
    proxy_mean: float
    proxy_se: float
    primary_mean: float
    primary_se: float
    # Difference (primary - proxy) estimate and variance
    est_diff: float
    est_diff_var: float
    # PPI rectifier components (when using cross-domain weights)
    rectifier: float | None = None
    rectifier_var: float | None = None
    rectified_proxy_mean: float | None = None
    # PPI error: primary - rectified_proxy
    ppi_error: float | None = None
    ppi_error_var: float | None = None



weighted = True

"""
Compute per-domain estimate statistics for all source domains.

For each domain (excluding target), computes:
- Basic stats: proxy_mean, primary_mean, and their SEs
- Difference estimate: primary - proxy and its variance
- PPI rectifier: weighted (primary - proxy) from OTHER source domains
- PPI error: primary - (proxy + rectifier)
- The weighted PPI error must be weighted to the specific target domain, hence the use of _cross_domain_weight_pattern

Handles missing values in primary/proxy columns by excluding them.

Args:
    weighted: If True, use importance weights for rectifier computation.

Returns:
    List of DomainEstimateStats for each source domain.
"""
domains = Adjusted_Estimator._get_sorted_domains()

# Source domains = all domains except target
if Adjusted_Estimator._target_domain_value is not None:
    domains_no_target = [d for d in domains if d != Adjusted_Estimator._target_domain_value]
else:
    domains_no_target = domains

# Pre-extract columns as numpy arrays (faster than repeated DataFrame access)
df = Adjusted_Estimator._df
domain_col = df[Adjusted_Estimator._domain_column].values
proxy_col = df[Adjusted_Estimator._proxy_outcome_column].values.astype(np.float64)
primary_col = df[Adjusted_Estimator._primary_outcome_column].values.astype(np.float64)

# Pre-compute source masks for PPI rectifier
cross_domain_weight_pattern = Adjusted_Estimator._cross_domain_weight_pattern
source_masks: dict[Any, npt.NDArray[np.bool_]] = {}
if cross_domain_weight_pattern is not None:
    for domain in domains_no_target:
        if Adjusted_Estimator._target_domain_value is not None:
            source_masks[domain] = (domain_col != domain) & (
                domain_col != Adjusted_Estimator._target_domain_value
            )
        else:
            source_masks[domain] = domain_col != domain

# Pre-extract weight columns if needed
weight_cols: dict[Any, npt.NDArray[np.float64]] = {}
if cross_domain_weight_pattern is not None and weighted:
    for domain in domains_no_target:
        weight_col_name = cross_domain_weight_pattern.format(
            target_domain=domain
        )
        if weight_col_name in df.columns:
            weight_cols[domain] = df[weight_col_name].values.astype(np.float64)

domain_stats = []

for domain in domains_no_target:
    # Use numpy boolean indexing instead of DataFrame filtering
    domain_mask = domain_col == domain
    proxy_raw = proxy_col[domain_mask]
    primary_raw = primary_col[domain_mask]

    # Get primary and proxy values, handling missing values
    proxy, primary = Adjusted_Estimator._get_valid_values(proxy_raw, primary_raw)
    n = len(proxy)

    if n < 2:
        continue

    # Basic statistics
    proxy_mean = float(np.mean(proxy))
    primary_mean = float(np.mean(primary))

    # Difference estimate (primary - proxy) and its variance
    # Compute variance and covariance in fewer passes
    est_diff = primary_mean - proxy_mean
    proxy_centered = proxy - proxy_mean
    primary_centered = primary - primary_mean
    var_primary = float(np.sum(primary_centered**2) / (n - 1) / n)
    var_proxy = float(np.sum(proxy_centered**2) / (n - 1) / n)
    proxy_se = float(np.sqrt(var_proxy))
    primary_se = float(np.sqrt(var_primary))
    cov_primary_proxy = float(
        np.sum(primary_centered * proxy_centered) / (n - 1) / n
    )
    est_diff_var = var_primary + var_proxy - 2 * cov_primary_proxy

    # Initialize PPI components
    rectifier: float | None = None
    rectifier_var: float | None = None
    rectified_proxy_mean: float | None = None
    ppi_error: float | None = None
    ppi_error_var: float | None = None

    # Compute PPI rectifier from OTHER source domains
    if cross_domain_weight_pattern is not None:
        # Use pre-computed source mask
        rect_mask = source_masks[domain]
        rect_proxy_raw = proxy_col[rect_mask]
        rect_primary_raw = primary_col[rect_mask]

        # Handle missing values in rectifier data
        rect_valid_mask = ~(
            np.isnan(rect_proxy_raw) | np.isnan(rect_primary_raw)
        )
        rect_proxy = rect_proxy_raw[rect_valid_mask]
        rect_primary = rect_primary_raw[rect_valid_mask]
        n_rect = len(rect_primary)

        if n_rect > 0:
            # Get weights for targeting the current domain
            if weighted and domain in weight_cols:
                weights_raw = weight_cols[domain][rect_mask]
                weights = weights_raw[rect_valid_mask]
            else:
                weights = np.ones(n_rect)

            # Normalize weights # Note, these should not be normalized a second tim.
            weights = weights / np.sum(weights)

            # Compute weighted rectifier
            d_values = rect_primary - rect_proxy
            domain_rectifier = float(np.sum(weights * d_values))
            residuals = d_values - domain_rectifier
            domain_rectifier_var = float(np.sum(weights**2 * residuals**2))

            # Rectified proxy and PPI error
            rectified_proxy_mean = proxy_mean + domain_rectifier
            ppi_error = primary_mean - rectified_proxy_mean
            ppi_error_var = (
                var_primary
                + var_proxy
                - 2 * cov_primary_proxy
                + domain_rectifier_var
            )

            rectifier = domain_rectifier
            rectifier_var = domain_rectifier_var

    domain_stats.append(
        DomainEstimateStats(
            domain=domain,
            n_samples=n,
            proxy_mean=proxy_mean,
            proxy_se=proxy_se,
            primary_mean=primary_mean,
            primary_se=primary_se,
            est_diff=est_diff,
            est_diff_var=max(est_diff_var, 1e-10),
            rectifier=rectifier,
            rectifier_var=rectifier_var,
            rectified_proxy_mean=rectified_proxy_mean,
            ppi_error=ppi_error,
            ppi_error_var=ppi_error_var,
        )
    )


In [ ]:
data

In [ ]:
generator = CovShiftDataGenerator(
    n_samples_per_domain=sample_size,
    n_domains=n_domains,
    concept_drift_degree=kappa,
    primary_lambda=PRIMARY_LAMBDA,
    primary_phi=PRIMARY_PHI,
    proxy_lambda=PROXY_LAMBDA,
    proxy_phi=PROXY_PHI,
    n_covariates=P,
    target_domain_means=mu_K,
    random_seed=seed,
)
data = generator.generate_data(include_cross_domain_weights = True)

In [ ]:
np.sum(data['weight_to_domain_1'])*(14/15)

In [ ]:
np.sum(data['importance_weight'])*(14/15)

In [ ]:
not_domain_1 = data['domain'] != 1
data_not_1 = data[not_domain_1]
data_1 = data[~not_domain_1]
w = data_not_1['weight_to_domain_1']

In [ ]:
primary = data_not_1['primary_outcome']
proxy = data_not_1['proxy_outcome']

proxy_1 = data_1['proxy_outcome']
w = w / np.sum(w)
d_values = primary - proxy
domain_rectifier = float(np.sum(w * d_values))
residuals = d_values - domain_rectifier


In [ ]:
np.mean(proxy_1)

In [ ]:
domain_rectifier

In [ ]:
np.mean(primary)

In [ ]:
np.mean(proxy)

In [ ]:
np.mean(proxy) + domain_rectifier

In [ ]:
domain_rectifier

In [ ]:
domain_rectifier_var = float(np.sum(w**2 * d_values**2))

In [ ]:
domain_rectifier_var

weighted_diff_means = data.groupby('domain').apply(lambda df: np.average(df['primary_outcome'] - df['proxy_outcome'], weights=df['importance_weight']))
weighted_diff_means

In [ ]:
# =============================================================================
# DEBUGGING CELL: Compare Confidence Intervals - Primary, Proxy Only, PPI-W
# with Hold-One-Domain-Out Validation Plot
# =============================================================================
# This cell visualizes and compares the confidence intervals from:
# 1. Primary metric (oracle baseline)
# 2. Proxy only metric
# 3. PPI-W (weighted Prediction-Powered Inference) estimate
# 4. Hold-one-domain-out validation showing primary and PPI-W for each held-out domain

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm


def debug_compare_confidence_intervals_with_hold_out(
    kappa: float = 0.00,
    sample_size: int = 100000,
    n_domains: int = 15,
    seed: int = 44,
):
    """
    Run a single simulation and visualize:
    1. Confidence intervals for Primary, Proxy Only, and PPI-W estimates
    2. Hold-one-domain-out validation plot showing primary and PPI-W for each held-out domain
    3. PPI errors with confidence intervals across domains
    """
    print("=" * 80)
    print("DEBUGGING: Confidence Interval Comparison with Hold-One-Out Validation")
    print("=" * 80)
    print(f"\nParameters:")
    print(f"  - kappa (concept drift): {kappa}")
    print(f"  - sample_size: {sample_size}")
    print(f"  - n_domains (K): {n_domains}")
    print(f"  - seed: {seed}")

    # Step 1: Generate data
    print("\n" + "-" * 80)
    print("STEP 1: Data Generation")
    print("-" * 80)

    generator = CovShiftDataGenerator(
        n_samples_per_domain=sample_size,
        n_domains=n_domains,
        concept_drift_degree=kappa,
        primary_lambda=PRIMARY_LAMBDA,
        primary_phi=PRIMARY_PHI,
        proxy_lambda=PROXY_LAMBDA,
        proxy_phi=PROXY_PHI,
        n_covariates=P,
        target_domain_means=mu_K,
        random_seed=seed,
    )
    data = generator.generate_data()

    print(f"  Generated {len(data)} total samples across {n_domains} domains")
    print(f"  Samples per domain: {data.groupby('domain').size().to_dict()}")

    # Compute true prevalence
    true_prev = generator.calculate_target_population_prevalence(n_samples=1_000_000)
    print(f"\n  True target prevalence: {true_prev:.6f}")

    # Step 2: Create estimator with cross_domain_weight_pattern
    print("\n" + "-" * 80)
    print("STEP 2: Create Estimator with Cross-Domain Weight Pattern")
    print("-" * 80)

    Adjusted_Estimator = AdjustedCovShiftPPIEstimator(
        df=data,
        primary_outcome_column="primary_outcome",
        proxy_outcome_column="proxy_outcome",
        importance_weight_column="importance_weight",
        domain_column="domain",
        target_domain_value=n_domains,
        cross_domain_weight_pattern='weight_to_domain_{target_domain}'
    )
    print("  Created AdjustedCovShiftPPIEstimator with cross_domain_weight_pattern")

    # Also create the base estimator for simpler estimates
    PPI_Estimator = CovShiftPPIEstimator(
        df=data,
        primary_outcome_column="primary_outcome",
        proxy_outcome_column="proxy_outcome",
        importance_weight_column="importance_weight",
        domain_column="domain",
        target_domain_value=n_domains,
    )

    # Step 3: Compute hold-one-domain-out estimate stats
    print("\n" + "-" * 80)
    print("STEP 3: Compute Hold-One-Domain-Out Estimate Stats")
    print("-" * 80)

    # Compute for both weighted and unweighted
    domain_stats_unweighted = Adjusted_Estimator.compute_hold_one_domain_out_estimate_stats(weighted=False)
    domain_stats_weighted = Adjusted_Estimator.compute_hold_one_domain_out_estimate_stats(weighted=True)

    print(f"\n  Computed stats for {len(domain_stats_unweighted)} source domains (unweighted)")
    print(f"  Computed stats for {len(domain_stats_weighted)} source domains (weighted)")

    # Step 4: Print domain statistics table
    print("\n" + "-" * 80)
    print("STEP 4: Per-Domain Statistics")
    print("-" * 80)

    print(f"\n  {'Domain':<8} {'N':<6} {'Proxy':>10} {'Primary':>10} {'Diff':>10} {'Rectifier':>12} {'Rect Proxy':>12} {'PPI Error':>10}")
    print("  " + "-" * 90)

    for stats in domain_stats_weighted:
        rectifier_str = f"{stats.rectifier:.6f}" if stats.rectifier is not None else "N/A"
        rect_proxy_str = f"{stats.rectified_proxy_mean:.6f}" if stats.rectified_proxy_mean is not None else "N/A"
        ppi_error_str = f"{stats.ppi_error:.6f}" if stats.ppi_error is not None else "N/A"
        print(f"  {stats.domain:<8} {stats.n_samples:<6} {stats.proxy_mean:>10.6f} {stats.primary_mean:>10.6f} "
              f"{stats.est_diff:>10.6f} {rectifier_str:>12} {rect_proxy_str:>12} {ppi_error_str:>10}")

    # Step 5: Compute base and adjusted estimates
    print("\n" + "-" * 80)
    print("STEP 5: Compute Estimates and Latent Parameters")
    print("-" * 80)

    # Primary (Oracle)
    primary_est = Adjusted_Estimator.compute_primary_mean_estimate(confidence_level=CONFIDENCE_LEVEL)
    print(f"\n  Primary (Oracle): {primary_est.estimate_val:.6f}")

    # Proxy Only
    proxy_est = Adjusted_Estimator.compute_proxy_mean_estimate(confidence_level=CONFIDENCE_LEVEL)
    print(f"  Proxy Only: {proxy_est.estimate_val:.6f}")

    # PPI (unweighted)
    ppi_est = Adjusted_Estimator.compute_ppi_mean_estimate(weighted=False, confidence_level=CONFIDENCE_LEVEL)
    print(f"  PPI (unweighted): {ppi_est.estimate_val:.6f}")

    # PPI-W (weighted)
    ppi_w_est = Adjusted_Estimator.compute_ppi_mean_estimate(weighted=True, confidence_level=CONFIDENCE_LEVEL)
    print(f"  PPI-W (weighted): {ppi_w_est.estimate_val:.6f}")
    print(f"    95% CI: [{ppi_w_est.lower_bound:.6f}, {ppi_w_est.upper_bound:.6f}]")

    # Adjusted estimates with latent parameters
    proxy_adj_est = Adjusted_Estimator.compute_adjusted_proxy_estimate(
        confidence_level=CONFIDENCE_LEVEL, method="MoM"
    )
    ppi_adj_est = Adjusted_Estimator.compute_adjusted_ppi_estimate(
        weighted=False, confidence_level=CONFIDENCE_LEVEL, method="MoM"
    )
    ppi_w_adj_est = Adjusted_Estimator.compute_adjusted_ppi_estimate(
        weighted=True, confidence_level=CONFIDENCE_LEVEL, method="MoM"
    )

    print(f"\n  Latent Model Parameters (MoM):")
    print(f"    Proxy+Adj:  rho = {proxy_adj_est.rho:+.6f}, gamma^2 = {proxy_adj_est.gamma_squared:.6f}")
    print(f"    PPI+Adj:    rho = {ppi_adj_est.rho:+.6f}, gamma^2 = {ppi_adj_est.gamma_squared:.6f}")
    print(f"    PPI-W+Adj:  rho = {ppi_w_adj_est.rho:+.6f}, gamma^2 = {ppi_w_adj_est.gamma_squared:.6f}")

    # Step 6: Extract data for plotting
    print("\n" + "-" * 80)
    print("STEP 6: Visualizations")
    print("-" * 80)

    domains = [s.domain for s in domain_stats_weighted]
    proxy_means = np.array([s.proxy_mean for s in domain_stats_weighted])
    proxy_ses = np.array([s.proxy_se for s in domain_stats_weighted])
    primary_means = np.array([s.primary_mean for s in domain_stats_weighted])
    primary_ses = np.array([s.primary_se for s in domain_stats_weighted])
    est_diffs = np.array([s.est_diff for s in domain_stats_weighted])
    est_diff_vars = np.array([s.est_diff_var for s in domain_stats_weighted])
    ppi_errors = np.array([s.ppi_error if s.ppi_error is not None else 0 for s in domain_stats_weighted])
    ppi_error_vars = np.array([s.ppi_error_var if s.ppi_error_var is not None else 0 for s in domain_stats_weighted])
    rectified_proxies = np.array([s.rectified_proxy_mean if s.rectified_proxy_mean is not None else s.proxy_mean for s in domain_stats_weighted])

    # Compute 95% CI half-widths (z = 1.96 for 95%)
    z = norm.ppf(0.975)
    proxy_ci = z * proxy_ses
    primary_ci = z * primary_ses
    ppi_error_ci = z * np.sqrt(ppi_error_vars)

    # ==========================================================================
    # PLOT 1: Confidence Interval Comparison - Primary, Proxy Only, PPI-W
    # ==========================================================================
    print("\n  Creating Plot 1: Confidence Interval Comparison...")

    fig1, ax1 = plt.subplots(figsize=(12, 5))

    methods = ["Primary (Oracle)", "Proxy Only", "PPI-W"]
    estimates = [primary_est.estimate_val, proxy_est.estimate_val, ppi_w_est.estimate_val]
    lowers = [primary_est.lower_bound, proxy_est.lower_bound, ppi_w_est.lower_bound]
    uppers = [primary_est.upper_bound, proxy_est.upper_bound, ppi_w_est.upper_bound]
    colors = ["darkorange", "steelblue", "green"]

    y_positions = np.arange(len(methods))

    for i, (method, est, lower, upper, color) in enumerate(zip(methods, estimates, lowers, uppers, colors)):
        error_lower = est - lower
        error_upper = upper - est
        ax1.errorbar(
            est, y_positions[i],
            xerr=[[error_lower], [error_upper]],
            fmt='o',
            color=color,
            markersize=12,
            capsize=8,
            capthick=2,
            elinewidth=2,
            label=f"{method}: {est:.4f} [{lower:.4f}, {upper:.4f}]"
        )

    ax1.axvline(x=true_prev, color='red', linestyle='--', linewidth=2, label=f"True Prevalence: {true_prev:.4f}")
    ax1.set_yticks(y_positions)
    ax1.set_yticklabels(methods, fontsize=12)
    ax1.set_xlabel("Estimate Value", fontsize=12)
    ax1.set_title(f"95% Confidence Interval Comparison\n(kappa={kappa}, n={sample_size}, K={n_domains})", fontsize=14)
    ax1.legend(loc="upper right", fontsize=10)
    ax1.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # ==========================================================================
    # PLOT 2: Hold-One-Domain-Out Validation - Primary and PPI-W for each domain
    # ==========================================================================
    print("\n  Creating Plot 2: Hold-One-Domain-Out Validation (Primary vs PPI-W)...")

    fig2, ax2 = plt.subplots(figsize=(14, 8))

    domain_labels = [f'Domain {d}' for d in domains]
    y_pos = np.arange(len(domains))
    offset = 0.15

    # Plot Primary estimates with CIs for each held-out domain
    ax2.errorbar(
        primary_means, y_pos + offset, xerr=primary_ci,
        fmt='s', color='darkorange', markersize=10, capsize=5, capthick=2, elinewidth=2,
        label=f'Primary Mean (per domain)'
    )

    # For PPI-W per domain, we use the rectified proxy (which is proxy + rectifier from other domains)
    # The rectified proxy is the PPI estimate for each held-out domain
    ax2.errorbar(
        rectified_proxies, y_pos - offset, xerr=proxy_ci,  # Using proxy SEs as approximation
        fmt='o', color='green', markersize=10, capsize=5, capthick=2, elinewidth=2,
        label=f'Rectified Proxy (PPI-W per domain)'
    )

    # Also show the proxy-only estimates
    ax2.plot(proxy_means, y_pos, 'd', color='steelblue', markersize=8, alpha=0.6, label='Proxy Mean (per domain)')

    # Add vertical lines for overall estimates
    ax2.axvline(x=true_prev, color='red', linestyle='--', linewidth=2, label=f'True Prevalence ({true_prev:.4f})')
    ax2.axvline(x=ppi_w_est.estimate_val, color='green', linestyle='-', linewidth=2, alpha=0.7, label=f'Overall PPI-W ({ppi_w_est.estimate_val:.4f})')
    ax2.axvline(x=primary_est.estimate_val, color='darkorange', linestyle='-', linewidth=2, alpha=0.7, label=f'Overall Primary ({primary_est.estimate_val:.4f})')

    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(domain_labels, fontsize=10)
    ax2.set_xlabel("Estimate Value", fontsize=12)
    ax2.set_ylabel("Held-Out Domain", fontsize=12)
    ax2.set_title(f"Hold-One-Domain-Out Validation: Primary vs PPI-W\n(kappa={kappa}, n={sample_size}, K={n_domains})", fontsize=14)
    ax2.legend(loc="best", fontsize=9)
    ax2.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()

    # ==========================================================================
    # PLOT 3: PPI Errors with Confidence Intervals
    # ==========================================================================
    print("\n  Creating Plot 3: PPI Errors with Confidence Intervals...")

    fig3, ax3 = plt.subplots(figsize=(12, 6))

    ci_lower = ppi_errors - ppi_error_ci
    ci_upper = ppi_errors + ppi_error_ci

    ax3.errorbar(
        domains, ppi_errors, yerr=ppi_error_ci,
        fmt='o', elinewidth=2, capsize=4, color='b', markersize=8,
        label='PPI Errors'
    )
    ax3.fill_between(domains, ci_lower, ci_upper, color='b', alpha=0.1)
    ax3.axhline(y=0, color='black', linestyle='-', linewidth=1)
    ax3.axhline(y=np.mean(ppi_errors), color='red', linestyle='--', linewidth=2, label=f'Mean PPI Error ({np.mean(ppi_errors):.4f})')

    ax3.set_title(f'PPI Errors with 95% Confidence Intervals\n(kappa={kappa}, n={sample_size}, K={n_domains})', fontsize=14)
    ax3.set_xlabel('Domain', fontsize=12)
    ax3.set_ylabel('PPI Error', fontsize=12)
    ax3.set_xticks(domains)
    ax3.grid(True, alpha=0.3)
    ax3.legend(loc='best', fontsize=10)
    plt.tight_layout()
    plt.show()

    # ==========================================================================
    # Summary Table
    # ==========================================================================
    print("\n" + "-" * 80)
    print("Summary: Estimates and 95% Confidence Intervals")
    print("-" * 80)
    print(f"\n  {'Method':<15} {'Estimate':>12} {'Lower':>12} {'Upper':>12} {'CI Width':>12} {'Covers True':>12}")
    print("  " + "-" * 75)

    for name, est in [("Primary", primary_est), ("Proxy Only", proxy_est), ("PPI-W", ppi_w_est)]:
        ci_width = est.upper_bound - est.lower_bound
        covers = est.lower_bound <= true_prev <= est.upper_bound
        covers_str = "Yes" if covers else "No"
        print(f"  {name:<15} {est.estimate_val:>12.6f} {est.lower_bound:>12.6f} {est.upper_bound:>12.6f} {ci_width:>12.6f} {covers_str:>12}")

    print(f"\n  True prevalence: {true_prev:.6f}")

    # Return results
    return {
        "true_prevalence": true_prev,
        "domain_stats_weighted": domain_stats_weighted,
        "estimates": {
            "primary": primary_est,
            "proxy": proxy_est,
            "ppi_w": ppi_w_est,
        },
        "adjusted_estimates": {
            "proxy_adj": proxy_adj_est,
            "ppi_adj": ppi_adj_est,
            "ppi_w_adj": ppi_w_adj_est,
        },
    }


# Run the debugging function with specified parameters
results = debug_compare_confidence_intervals_with_hold_out(
    kappa=0.00,
    sample_size=100000,
    n_domains=15,
    seed=44,
)


In [ ]:
"""
Run a single simulation and visualize:
1. Confidence intervals for Primary, Proxy Only, and PPI-W estimates
2. Hold-one-domain-out validation plot showing primary and PPI-W for each held-out domain
3. PPI errors with confidence intervals across domains
"""


kappa: float = 0.00

sample_size: int = 100000

n_domains: int = 15

seed: int = 44




print("=" * 80)
print("DEBUGGING: Confidence Interval Comparison with Hold-One-Out Validation")
print("=" * 80)
print(f"\nParameters:")
print(f"  - kappa (concept drift): {kappa}")
print(f"  - sample_size: {sample_size}")
print(f"  - n_domains (K): {n_domains}")
print(f"  - seed: {seed}")

# Step 1: Generate data
print("\n" + "-" * 80)
print("STEP 1: Data Generation")
print("-" * 80)

generator = CovShiftDataGenerator(
    n_samples_per_domain=sample_size,
    n_domains=n_domains,
    concept_drift_degree=kappa,
    primary_lambda=PRIMARY_LAMBDA,
    primary_phi=PRIMARY_PHI,
    proxy_lambda=PROXY_LAMBDA,
    proxy_phi=PROXY_PHI,
    n_covariates=P,
    target_domain_means=mu_K,
    random_seed=seed,
)
data = generator.generate_data(include_cross_domain_weights=True)

print(f"  Generated {len(data)} total samples across {n_domains} domains")
print(f"  Samples per domain: {data.groupby('domain').size().to_dict()}")

# Compute true prevalence
true_prev = generator.calculate_target_population_prevalence(n_samples=1_000_000)
print(f"\n  True target prevalence: {true_prev:.6f}")

# Step 2: Create estimator with cross_domain_weight_pattern
print("\n" + "-" * 80)
print("STEP 2: Create Estimator with Cross-Domain Weight Pattern")
print("-" * 80)

Adjusted_Estimator = AdjustedCovShiftPPIEstimator(
    df=data,
    primary_outcome_column="primary_outcome",
    proxy_outcome_column="proxy_outcome",
    importance_weight_column="importance_weight",
    domain_column="domain",
    target_domain_value=n_domains,
    cross_domain_weight_pattern='weight_to_domain_{target_domain}'
)
print("  Created AdjustedCovShiftPPIEstimator with cross_domain_weight_pattern")

# Also create the base estimator for simpler estimates
PPI_Estimator = CovShiftPPIEstimator(
    df=data,
    primary_outcome_column="primary_outcome",
    proxy_outcome_column="proxy_outcome",
    importance_weight_column="importance_weight",
    domain_column="domain",
    target_domain_value=n_domains,
)

# Step 3: Compute hold-one-domain-out estimate stats
print("\n" + "-" * 80)
print("STEP 3: Compute Hold-One-Domain-Out Estimate Stats")
print("-" * 80)

# Compute for both weighted and unweighted
domain_stats_unweighted = Adjusted_Estimator.compute_hold_one_domain_out_estimate_stats(weighted=False)
domain_stats_weighted = Adjusted_Estimator.compute_hold_one_domain_out_estimate_stats(weighted=True)

print(f"\n  Computed stats for {len(domain_stats_unweighted)} source domains (unweighted)")
print(f"  Computed stats for {len(domain_stats_weighted)} source domains (weighted)")

# Step 4: Print domain statistics table
print("\n" + "-" * 80)
print("STEP 4: Per-Domain Statistics")
print("-" * 80)

print(f"\n  {'Domain':<8} {'N':<6} {'Proxy':>10} {'Primary':>10} {'Diff':>10} {'Rectifier':>12} {'Rect Proxy':>12} {'PPI Error':>10}")
print("  " + "-" * 90)

for stats in domain_stats_weighted:
    rectifier_str = f"{stats.rectifier:.6f}" if stats.rectifier is not None else "N/A"
    rect_proxy_str = f"{stats.rectified_proxy_mean:.6f}" if stats.rectified_proxy_mean is not None else "N/A"
    ppi_error_str = f"{stats.ppi_error:.6f}" if stats.ppi_error is not None else "N/A"
    print(f"  {stats.domain:<8} {stats.n_samples:<6} {stats.proxy_mean:>10.6f} {stats.primary_mean:>10.6f} "
            f"{stats.est_diff:>10.6f} {rectifier_str:>12} {rect_proxy_str:>12} {ppi_error_str:>10}")

# Step 5: Compute base and adjusted estimates
print("\n" + "-" * 80)
print("STEP 5: Compute Estimates and Latent Parameters")
print("-" * 80)

# Primary (Oracle)
primary_est = Adjusted_Estimator.compute_primary_mean_estimate(confidence_level=CONFIDENCE_LEVEL)
print(f"\n  Primary (Oracle): {primary_est.estimate_val:.6f}")

# Proxy Only
proxy_est = Adjusted_Estimator.compute_proxy_mean_estimate(confidence_level=CONFIDENCE_LEVEL)
print(f"  Proxy Only: {proxy_est.estimate_val:.6f}")

# PPI (unweighted)
ppi_est = Adjusted_Estimator.compute_ppi_mean_estimate(weighted=False, confidence_level=CONFIDENCE_LEVEL)
print(f"  PPI (unweighted): {ppi_est.estimate_val:.6f}")

# PPI-W (weighted)
ppi_w_est = Adjusted_Estimator.compute_ppi_mean_estimate(weighted=True, confidence_level=CONFIDENCE_LEVEL)
print(f"  PPI-W (weighted): {ppi_w_est.estimate_val:.6f}")
print(f"    95% CI: [{ppi_w_est.lower_bound:.6f}, {ppi_w_est.upper_bound:.6f}]")

# Adjusted estimates with latent parameters
proxy_adj_est = Adjusted_Estimator.compute_adjusted_proxy_estimate(
    confidence_level=CONFIDENCE_LEVEL, method="MoM"
)
ppi_adj_est = Adjusted_Estimator.compute_adjusted_ppi_estimate(
    weighted=False, confidence_level=CONFIDENCE_LEVEL, method="MoM"
)
ppi_w_adj_est = Adjusted_Estimator.compute_adjusted_ppi_estimate(
    weighted=True, confidence_level=CONFIDENCE_LEVEL, method="MoM"
)

print(f"\n  Latent Model Parameters (MoM):")
print(f"    Proxy+Adj:  rho = {proxy_adj_est.rho:+.6f}, gamma^2 = {proxy_adj_est.gamma_squared:.6f}")
print(f"    PPI+Adj:    rho = {ppi_adj_est.rho:+.6f}, gamma^2 = {ppi_adj_est.gamma_squared:.6f}")
print(f"    PPI-W+Adj:  rho = {ppi_w_adj_est.rho:+.6f}, gamma^2 = {ppi_w_adj_est.gamma_squared:.6f}")

# Step 6: Extract data for plotting
print("\n" + "-" * 80)
print("STEP 6: Visualizations")
print("-" * 80)

In [ ]:
domains = [s.domain for s in domain_stats_weighted]
proxy_means = np.array([s.proxy_mean for s in domain_stats_weighted])
proxy_ses = np.array([s.proxy_se for s in domain_stats_weighted])
primary_means = np.array([s.primary_mean for s in domain_stats_weighted])
primary_ses = np.array([s.primary_se for s in domain_stats_weighted])
est_diffs = np.array([s.est_diff for s in domain_stats_weighted])
est_diff_vars = np.array([s.est_diff_var for s in domain_stats_weighted])
ppi_errors = np.array([s.ppi_error if s.ppi_error is not None else 0 for s in domain_stats_weighted])
ppi_error_vars = np.array([s.ppi_error_var if s.ppi_error_var is not None else 0 for s in domain_stats_weighted])
# rectified_proxies = np.array([s.rectified_proxy_mean if s.rectified_proxy_mean is not None else s.proxy_mean for s in domain_stats_weighted])
ppi_mean = np.array([s.ppi_mean if s.ppi_mean is not None else 0 for s in domain_stats_weighted])
ppi_se = np.array([s.ppi_se if s.ppi_se is not None else 0 for s in domain_stats_weighted])


# Compute 95% CI half-widths (z = 1.96 for 95%)
z = norm.ppf(0.975)
proxy_ci = z * proxy_ses
primary_ci = z * primary_ses
ppi_error_ci = z * np.sqrt(ppi_error_vars)
ppi_mean_ci = z * ppi_se

In [ ]:
primary_means

In [ ]:
primary_ses

In [ ]:
est_diffs


In [ ]:
# ==========================================================================
# PLOT 1: Confidence Interval Comparison - Primary, Proxy Only, PPI-W
# ==========================================================================
print("\n  Creating Plot 1: Confidence Interval Comparison...")

fig1, ax1 = plt.subplots(figsize=(12, 5))

methods = ["Primary (Oracle)", "Proxy Only", "PPI-W"]
estimates = [primary_est.estimate_val, proxy_est.estimate_val, ppi_w_est.estimate_val]
lowers = [primary_est.lower_bound, proxy_est.lower_bound, ppi_w_est.lower_bound]
uppers = [primary_est.upper_bound, proxy_est.upper_bound, ppi_w_est.upper_bound]
colors = ["darkorange", "steelblue", "green"]

y_positions = np.arange(len(methods))

for i, (method, est, lower, upper, color) in enumerate(zip(methods, estimates, lowers, uppers, colors)):
    error_lower = est - lower
    error_upper = upper - est
    ax1.errorbar(
        est, y_positions[i],
        xerr=[[error_lower], [error_upper]],
        fmt='o',
        color=color,
        markersize=12,
        capsize=8,
        capthick=2,
        elinewidth=2,
        label=f"{method}: {est:.4f} [{lower:.4f}, {upper:.4f}]"
    )

ax1.axvline(x=true_prev, color='red', linestyle='--', linewidth=2, label=f"True Prevalence: {true_prev:.4f}")
ax1.set_yticks(y_positions)
ax1.set_yticklabels(methods, fontsize=12)
ax1.set_xlabel("Estimate Value", fontsize=12)
ax1.set_title(f"95% Confidence Interval Comparison\n(kappa={kappa}, n={sample_size}, K={n_domains})", fontsize=14)
ax1.legend(loc="upper right", fontsize=10)
ax1.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
#


In [ ]:
# ==========================================================================
# PLOT 2: Hold-One-Domain-Out Validation - Primary and PPI-W for each domain
# ==========================================================================
print("\n  Creating Plot 2: Hold-One-Domain-Out Validation (Primary vs PPI-W)...")

fig2, ax2 = plt.subplots(figsize=(14, 8))

domain_labels = [f'Domain {d}' for d in domains]
y_pos = np.arange(len(domains))
offset = 0.15

# Plot Primary estimates with CIs for each held-out domain
ax2.errorbar(
    primary_means, y_pos + offset, xerr=primary_ci,
    fmt='s', color='darkorange', markersize=10, capsize=5, capthick=2, elinewidth=2,
    label=f'Primary Mean (per domain)'
)

# For PPI-W per domain, we use the rectified proxy (which is proxy + rectifier from other domains)
# The rectified proxy is the PPI estimate for each held-out domain
ax2.errorbar(
    ppi_mean, y_pos - offset, xerr=ppi_mean_ci,  # Using proxy SEs as approximation
    fmt='o', color='green', markersize=10, capsize=5, capthick=2, elinewidth=2,
    label=f'Rectified Proxy (PPI-W per domain)'
)

# Also show the proxy-only estimates
ax2.plot(proxy_means, y_pos, 'd', color='steelblue', markersize=8, alpha=0.6, label='Proxy Mean (per domain)')

# Add vertical lines for overall estimates
ax2.axvline(x=true_prev, color='red', linestyle='--', linewidth=2, label=f'True Prevalence ({true_prev:.4f})')
ax2.axvline(x=ppi_w_est.estimate_val, color='green', linestyle='-', linewidth=2, alpha=0.7, label=f'Overall PPI-W ({ppi_w_est.estimate_val:.4f})')
ax2.axvline(x=primary_est.estimate_val, color='darkorange', linestyle='-', linewidth=2, alpha=0.7, label=f'Overall Primary ({primary_est.estimate_val:.4f})')

ax2.set_yticks(y_pos)
ax2.set_yticklabels(domain_labels, fontsize=10)
ax2.set_xlabel("Estimate Value", fontsize=12)
ax2.set_ylabel("Held-Out Domain", fontsize=12)
ax2.set_title(f"Hold-One-Domain-Out Validation: Primary vs PPI-W\n(kappa={kappa}, n={sample_size}, K={n_domains})", fontsize=14)
ax2.legend(loc="best", fontsize=9)
ax2.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================================
# PLOT 3: PPI Errors with Confidence Intervals
# ==========================================================================
print("\n  Creating Plot 3: PPI Errors with Confidence Intervals...")

fig3, ax3 = plt.subplots(figsize=(12, 6))

ci_lower = ppi_errors - ppi_error_ci
ci_upper = ppi_errors + ppi_error_ci

ax3.errorbar(
    domains, ppi_errors, yerr=ppi_error_ci,
    fmt='o', elinewidth=2, capsize=4, color='b', markersize=8,
    label='PPI Errors'
)
ax3.fill_between(domains, ci_lower, ci_upper, color='b', alpha=0.1)
ax3.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax3.axhline(y=np.mean(ppi_errors), color='red', linestyle='--', linewidth=2, label=f'Mean PPI Error ({np.mean(ppi_errors):.4f})')

ax3.set_title(f'PPI Errors with 95% Confidence Intervals\n(kappa={kappa}, n={sample_size}, K={n_domains})', fontsize=14)
ax3.set_xlabel('Domain', fontsize=12)
ax3.set_ylabel('PPI Error', fontsize=12)
ax3.set_xticks(domains)
ax3.grid(True, alpha=0.3)
ax3.legend(loc='best', fontsize=10)
plt.tight_layout()
plt.show()

# ==========================================================================
# Summary Table
# ==========================================================================
print("\n" + "-" * 80)
print("Summary: Estimates and 95% Confidence Intervals")
print("-" * 80)
print(f"\n  {'Method':<15} {'Estimate':>12} {'Lower':>12} {'Upper':>12} {'CI Width':>12} {'Covers True':>12}")
print("  " + "-" * 75)

for name, est in [("Primary", primary_est), ("Proxy Only", proxy_est), ("PPI-W", ppi_w_est)]:
    ci_width = est.upper_bound - est.lower_bound
    covers = est.lower_bound <= true_prev <= est.upper_bound
    covers_str = "Yes" if covers else "No"
    print(f"  {name:<15} {est.estimate_val:>12.6f} {est.lower_bound:>12.6f} {est.upper_bound:>12.6f} {ci_width:>12.6f} {covers_str:>12}")

print(f"\n  True prevalence: {true_prev:.6f}")

# Return results
return {
    "true_prevalence": true_prev,
    "domain_stats_weighted": domain_stats_weighted,
    "estimates": {
        "primary": primary_est,
        "proxy": proxy_est,
        "ppi_w": ppi_w_est,
    },
    "adjusted_estimates": {
        "proxy_adj": proxy_adj_est,
        "ppi_adj": ppi_adj_est,
        "ppi_w_adj": ppi_w_adj_est,
    },
}


# Run the debugging function with specified parameters
results = debug_compare_confidence_intervals_with_hold_out(
    kappa=0.00,
    sample_size=100000,
    n_domains=15,
    seed=44,
)

In [ ]:
# Manual domain k error
data

target_domain = 15
hold_out_k = 2

data_target = data[data.domain == target_domain]
data_others = data[data.domain != target_domain]

data_k = data_others[data_others['domain'] == hold_out_k]

data_not_k = data_others[data_others['domain'] != hold_out_k]


In [ ]:
primary_k = data_k['primary_outcome'].mean()
primary_k_se = data_k['primary_outcome'].std() / np.sqrt(len(data_k))


In [ ]:
proxy_k = data_k['proxy_outcome'].mean()
proxy_k_se = data_k['proxy_outcome'].std() / np.sqrt(len(data_k))

ppi_k = data_k['importance_weight'].mean() * (proxy_k - primary_k)
ppi_k_se = np.sqrt(
    primary_k_se**2 + proxy_k_se**2
)

ppi_k_adj = ppi_k + (primary_k - proxy_k)
ppi_k_adj_se = np.sqrt(

In [ ]:
primary_k

# primary_k_se

In [ ]:
data_not_k